In [ ]:
# ============================================================
# Cell 0 — Google Drive Persistence (run BEFORE training)
# ============================================================
"""
Mount Google Drive and set up a save directory so that checkpoints,
plots, metrics, and the notebook itself are backed up automatically.

Run this cell BEFORE starting training.  If the session dies, your
trained weights and saved figures will still be on your Drive.
"""
import shutil, os, json, numpy as np

# ── 0. HuggingFace Token (speeds up dataset download, avoids rate limits) ──
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')  # Set via Colab Secrets → "HF_TOKEN"
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    print(f"✓ HuggingFace token loaded ({len(HF_TOKEN)} chars)")
else:
    print("ℹ No HF_TOKEN found — downloads may be rate-limited.")
    print("  Add it via: Colab menu → Runtime → Manage secrets → HF_TOKEN")


# ── 1. Mount Drive ───────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
    print("Not running in Colab — skipping Drive mount. "
          "Files will be saved locally only.")

# ── 2. Create persistent output directory ────────────────────
PROJECT_NAME = "RIR_Project"
if IN_COLAB:
    SAVE_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}_outputs"
else:
    SAVE_DIR = os.path.join(os.getcwd(), f"{PROJECT_NAME}_outputs")

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")


# ── 3. Helper functions for saving ───────────────────────────

def save_checkpoint(state_dict: dict, name: str) -> str:
    """Save a model checkpoint to the persistent directory."""
    path = os.path.join(SAVE_DIR, name)
    torch.save(state_dict, path)
    print(f"  💾 Checkpoint saved → {path}")
    return path


def save_figure(fig_or_path, name: str) -> str:
    """Save a matplotlib figure or copy an existing file to Drive."""
    dest = os.path.join(SAVE_DIR, name)
    if isinstance(fig_or_path, str) and os.path.exists(fig_or_path):
        shutil.copy2(fig_or_path, dest)
    else:
        # Assume it's a matplotlib figure
        fig_or_path.savefig(dest, dpi=150, bbox_inches="tight")
    print(f"  📊 Figure saved → {dest}")
    return dest


def save_metrics(metrics_dict: dict, name: str = "test_metrics.json") -> str:
    """Save evaluation metrics as JSON."""
    path = os.path.join(SAVE_DIR, name)
    with open(path, "w") as f:
        json.dump(metrics_dict, f, indent=2, default=str)
    print(f"  📋 Metrics saved → {path}")
    return path


def save_history(history: dict, name: str = "training_history.json") -> str:
    """Save training history as JSON."""
    path = os.path.join(SAVE_DIR, name)
    with open(path, "w") as f:
        json.dump(history, f, indent=2)
    print(f"  📈 History saved → {path}")
    return path


def save_rir_audio(
    rir: np.ndarray,
    sample_rate: int = 16_000,
    name: str = "generated_rir.wav",
) -> str:
    """Save a generated RIR as a .wav file."""
    from scipy.io import wavfile
    path = os.path.join(SAVE_DIR, name)
    # Normalise to prevent clipping
    rir_norm = rir / (np.abs(rir).max() + 1e-30) * 0.95
    wavfile.write(path, sample_rate, rir_norm.astype(np.float32))
    print(f"  🔊 Audio saved → {path}")
    return path


def backup_notebook(notebook_name: str = "RIR_Project.ipynb") -> str:
    """Copy the current notebook to Drive for safekeeping."""
    if IN_COLAB:
        src = f"/content/{notebook_name}"
        if not os.path.exists(src):
            # Try the working directory
            src = notebook_name
        if os.path.exists(src):
            dest = os.path.join(SAVE_DIR, notebook_name)
            shutil.copy2(src, dest)
            print(f"  📓 Notebook backed up → {dest}")
            return dest
        else:
            print(f"  ⚠ Could not find {notebook_name} to back up")
            return ""
    else:
        print("  ℹ Notebook backup only works in Colab")
        return ""


print(f"\n✓ Persistence helpers ready.  All outputs will save to:")
print(f"  {SAVE_DIR}")
print(f"\nFiles that will be saved after training:")
print(f"  • best_lstm.pt, best_fdn.pt       — model checkpoints")
print(f"  • training_history.json            — per-epoch loss values")
print(f"  • test_metrics.json                — RT60/LSD/EDC/DRR scores")
print(f"  • training_curves.png              — loss plots")
print(f"  • rir_sample_*.png                 — waveform comparisons")
print(f"  • generated_rir_*.wav              — audio files you can listen to")
print(f"  • RIR_Project.ipynb                — notebook backup")

# Physics-Informed RIR Generation Framework

> **Notebook goal:** Build an end-to-end generative pipeline that maps macroscopic room parameters to realistic Room Impulse Responses (RIRs), guided by acoustic physics priors.

---

## Phase 1 — Data Pipeline

**Dataset:** HuggingFace [`mandipgoswami/rirmega`](https://huggingface.co/datasets/mandipgoswami/rirmega) (4 000 measured RIRs)

### Tensor Schema

| Tensor | Shape | Contents |
|:-------|:-----:|:---------|
| **X** *(input)* | `[16]` | `[L, W, H, src_x, src_y, src_z, mic_x, mic_y, mic_z, a_broad, a_125, a_250, a_500, a_1k, a_2k, a_4k]` |
| **Y["metrics"]** | `[10]` | `[RT60, DRR_dB, C50_dB, C80_dB, band_RT60_125, ..., band_RT60_4k]` |
| **Y["rir"]** | `[max_len]` | Padded / truncated RIR waveform (32 000 samples @ 16 kHz = 2 s) |
| **Y["edc"]** | `[max_len]` | Schroeder backward-integration Energy Decay Curve (dB) |
| **Y["rir_length"]** | scalar | Original sample count before pad/truncate |

### Pipeline Flow

```
HuggingFace AudioFolder --> decode audio --> pad/truncate --> RIR tensor
           |                                                      |
   metadata CSV (hf_hub_download)                          Schroeder EDC
           |                                                      |
     parse JSON fields --> build X[16] ----------------> (X, Y) Dataset
```

## Related Works & Motivation

### Why Generate RIRs Synthetically?

Measuring Room Impulse Responses (RIRs) requires specialised equipment (omni-directional loudspeakers, microphone arrays, swept-sine or MLS signals) and **hours of setup per room configuration**. A single room with 10 source–receiver pairs already demands significant measurement effort; scaling to thousands of rooms for robust training of downstream speech/audio models is prohibitively expensive.

**Our approach:** Train a physics-informed neural generative model that maps compact room descriptors (geometry, absorption, source/mic positions, acoustic mode features) to full-length RIR waveforms—replacing costly measurements with learned synthesis.

---

### Related Work

**1. Raissi et al. — *Physics-Informed Neural Networks* (2019)**
* **Key Idea:** Embed PDE residuals (Navier–Stokes, Schrödinger, etc.) directly into the loss, enabling neural nets to respect known physics without labelled PDE solutions.
* **Limitation Addressed:** Our work applies the PINN paradigm to **room acoustics**: we penalise violations of the acoustic continuity and linearised momentum equations, ensuring the predicted energy decay obeys wave-propagation physics.

**2. Ratnarajah et al. — *FAST-RIR* (2022)**
* **Key Idea:** Use a GAN conditioned on room parameters to generate late-reverb RIRs in the spectrogram domain, then invert to time domain via Griffin–Lim.
* **Limitation Addressed:** FAST-RIR ignores explicit physical constraints. Our physics-informed loss and differentiable FDN enforce acoustic consistency that a pure GAN cannot guarantee, yielding more physically plausible decay curves.

**3. Schlecht & Habets — *On Lossless Feedback Delay Networks* (2017)**
* **Key Idea:** Formalise the Feedback Delay Network (FDN)—parallel delay lines + unitary feedback matrix—as the standard parametric model for artificial late reverberation.
* **Limitation Addressed:** We make the FDN **fully differentiable** (learnable delays, per-channel absorption, Hadamard feedback) and train it jointly with an LSTM predictor via gradient descent, rather than hand-tuning parameters.

**4. Steinmetz et al. — *Filtered Noise Shaping* (2021)**
* **Key Idea:** Learn to estimate RIR parameters (RT60, DRR) from reverberant speech and reconstruct plausible RIRs using filtered-noise shaping.
* **Limitation Addressed:** Their approach is analysis $\rightarrow$ resynthesis from an observed signal. Ours is **parameter $\rightarrow$ generation**: given room geometry + absorption, we synthesise a novel RIR without any audio observation.

---

### Our Innovation (Contributions)

1.  **Physics-Informed Generative Pipeline** — First application of PINN-style acoustic PDE losses (continuity + Euler momentum) to neural RIR generation.
2.  **Room Acoustic Mode Features** — 8-dimensional modal feature vector extracted from room geometry, giving the model explicit low-frequency physics cues.
3.  **Differentiable FDN with Learned Delays** — End-to-end trainable FDN whose delay lengths and absorption parameters are optimised via back-propagation.
4.  **4-Phase Curriculum Training** — Warmup (EDC only) $\rightarrow$ Physics ramp $\rightarrow$ FDN co-training $\rightarrow$ Fine-tune schedule that stabilises convergence.

In [ ]:
# ============================================================
# Cell 3 — Install Dependencies (Colab)
# ============================================================
!pip install -q datasets huggingface_hub torch torchaudio pandas numpy scipy matplotlib

In [ ]:
# ============================================================
# Cell 4 — Imports & Constants
# ============================================================
from __future__ import annotations

import ast
import json
import os
import re
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download

# --- Constants ---
OCTAVE_BANDS: List[str] = ["125", "250", "500", "1000", "2000", "4000"]
DEFAULT_MAX_RIR_LEN: int = 32_000          # 2 s @ 16 kHz
MODAL_FEAT_DIM: int = 8                    # room acoustic mode features
INPUT_DIM: int = 16 + MODAL_FEAT_DIM       # 3 room + 3 src + 3 mic + 1 abs + 6 band abs + 8 modal
METRICS_DIM: int = 10                      # 4 scalar metrics + 6 band RT60s

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# ============================================================
# Cell 5 — Helper Functions
# ============================================================

def _parse_json_field(val):
    """Robustly parse a CSV cell that stores JSON / Python-literal data."""
    if isinstance(val, (dict, list)):
        return val
    if pd.isna(val) or val is None:
        return None
    try:
        return json.loads(val.replace("'", '"'))
    except (json.JSONDecodeError, AttributeError):
        pass
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return None


def compute_edc(rir: np.ndarray) -> np.ndarray:
    """Schroeder backward-integration Energy Decay Curve (dB).

    EDC(t) = integral from t to infinity of h^2(tau) dtau
    Normalised so EDC[0] = 0 dB.
    """
    h2 = rir.astype(np.float64) ** 2
    edc = np.cumsum(h2[::-1])[::-1].copy()
    edc /= edc[0] + 1e-30                    # normalise
    edc_db = 10.0 * np.log10(edc + 1e-30)
    return edc_db.astype(np.float32)


def _extract_sample_id(path: str) -> Optional[str]:
    """Pull 'rir_NNNNNN' out of an audio file path."""
    if not path:
        return None
    m = re.search(r"(rir_\d+)", os.path.basename(str(path)))
    return m.group(1) if m else None


def compute_multiband_edc(
    rir: np.ndarray,
    sr: int = 16_000,
    bands: List[str] = None,
    num_time_steps: int = 256,
) -> np.ndarray:
    """Compute multiband Energy Decay Curves from a broadband RIR.

    Filters the RIR into octave bands using 4th-order Butterworth
    bandpass filters, computes Schroeder EDC per band, then
    downsamples to ``num_time_steps``.

    Parameters
    ----------
    rir : ndarray[L]
        Raw RIR waveform.
    sr : int
        Sample rate in Hz.
    bands : list[str]
        Octave-band centre frequencies as strings (default: OCTAVE_BANDS).
    num_time_steps : int
        Temporal resolution of the output EDC.

    Returns
    -------
    edc_mb : ndarray[num_time_steps, num_bands]
        Multiband EDC in dB.
    """
    from scipy.signal import butter, sosfilt

    if bands is None:
        bands = OCTAVE_BANDS

    nyq = sr / 2.0
    L = len(rir)
    edcs = []

    for fc_str in bands:
        fc = float(fc_str)
        lo = fc / np.sqrt(2.0)
        hi = fc * np.sqrt(2.0)
        # Clamp to Nyquist
        lo = max(lo, 20.0)
        hi = min(hi, nyq - 1.0)

        sos = butter(4, [lo / nyq, hi / nyq], btype="band", output="sos")
        filtered = sosfilt(sos, rir).astype(np.float32)
        edc_band = compute_edc(filtered)
        edcs.append(edc_band)

    # Stack → [L, num_bands]
    edc_full = np.stack(edcs, axis=1)

    # Downsample from L → num_time_steps via strided indexing
    indices = np.linspace(0, L - 1, num_time_steps, dtype=int)
    return edc_full[indices]  # [num_time_steps, num_bands]


def downsample_edc_tensor(
    edc: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    """Downsample EDC tensor [B, L] → [B, num_steps] via adaptive avg pooling."""
    return torch.nn.functional.adaptive_avg_pool1d(
        edc.unsqueeze(1), num_steps
    ).squeeze(1)


# ------------------------------------------------------------------
#  Room Acoustic Mode Features
# ------------------------------------------------------------------

def compute_room_modes(
    L: float, W: float, H: float,
    c: float = 343.0,
    max_order: int = 6,
) -> np.ndarray:
    """Compute room acoustic mode features from rectangular room dimensions.

    For a rectangular room, standing-wave (modal) frequencies are:
      f(nx,ny,nz) = (c/2) * sqrt((nx/L)^2 + (ny/W)^2 + (nz/H)^2)

    Returns an 8-dimensional feature vector:
      [0] : Number of modes below 300 Hz (low-frequency density)
      [1] : Schroeder frequency  f_s = 2000 * sqrt(T60_est / V)
            approximated from room volume as T60_est ~ 0.16 * V / (S * 0.2)
      [2] : 1st axial mode frequency (Hz)
      [3] : Mean axial mode spacing (Hz)
      [4] : Std of axial mode spacing (Hz) -- irregularity measure
      [5] : Modal overlap factor at 500 Hz (modes per bandwidth)
      [6] : Ratio of tangential to axial mode count below 500 Hz
      [7] : log10(room volume) for scale normalisation
    """
    dims = sorted([L, W, H], reverse=True)  # ensure L >= W >= H
    L_, W_, H_ = dims
    V = L_ * W_ * H_

    # Enumerate all modes up to max_order
    axial, tangential, oblique = [], [], []
    all_modes = []
    for nx in range(0, max_order + 1):
        for ny in range(0, max_order + 1):
            for nz in range(0, max_order + 1):
                if nx == 0 and ny == 0 and nz == 0:
                    continue
                f = (c / 2.0) * np.sqrt(
                    (nx / L_) ** 2 + (ny / W_) ** 2 + (nz / H_) ** 2
                )
                all_modes.append(f)
                nonzero = (nx > 0) + (ny > 0) + (nz > 0)
                if nonzero == 1:
                    axial.append(f)
                elif nonzero == 2:
                    tangential.append(f)
                else:
                    oblique.append(f)

    all_modes = sorted(all_modes)
    axial = sorted(axial)
    tangential = sorted(tangential)

    # [0] Mode count below 300 Hz
    n_below_300 = sum(1 for f in all_modes if f < 300.0)

    # [1] Schroeder frequency approximation
    S = 2.0 * (L_ * W_ + W_ * H_ + L_ * H_)  # surface area
    avg_alpha = 0.2                             # typical absorption
    t60_est = 0.161 * V / (S * avg_alpha + 1e-9)
    f_schroeder = 2000.0 * np.sqrt(t60_est / (V + 1e-9))

    # [2] 1st axial mode
    f_first_axial = axial[0] if axial else 0.0

    # [3-4] Axial mode spacing stats
    if len(axial) >= 2:
        spacings = np.diff(axial)
        mean_spacing = float(np.mean(spacings))
        std_spacing = float(np.std(spacings))
    else:
        mean_spacing = 0.0
        std_spacing = 0.0

    # [5] Modal overlap factor at 500 Hz
    #   M(f) = n(f) * delta_f,  where n(f) = 4*pi*f^2*V/c^3 + pi*f*S/(2*c^2)
    #   and delta_f = 2.2 / T60
    f_ref = 500.0
    n_density = (4 * np.pi * f_ref**2 * V / c**3
                 + np.pi * f_ref * S / (2 * c**2))
    delta_f = 2.2 / (t60_est + 1e-9)
    modal_overlap = n_density * delta_f

    # [6] Tangential-to-axial ratio below 500 Hz
    n_axial_500 = sum(1 for f in axial if f < 500.0)
    n_tang_500 = sum(1 for f in tangential if f < 500.0)
    tang_axial_ratio = n_tang_500 / (n_axial_500 + 1e-9)

    # [7] log volume
    log_volume = np.log10(V + 1e-9)

    return np.array([
        n_below_300,
        f_schroeder,
        f_first_axial,
        mean_spacing,
        std_spacing,
        modal_overlap,
        tang_axial_ratio,
        log_volume,
    ], dtype=np.float32)


print("Helper functions defined (including room acoustic mode features).")

In [ ]:
# ============================================================
# Cell 6 — RIRMegaDataset Class
# ============================================================

class RIRMegaDataset(Dataset):
    """PyTorch Dataset for the RIRmega corpus.

    Parameters
    ----------
    split : str
        One of "train", "val" (or "validation"), "test".
    max_rir_len : int
        Fixed waveform length in samples (pad / truncate).
    cache_dir : str | None
        Optional HuggingFace cache directory.
    """

    _HF_SPLIT_MAP = {
        "train": "train", "validation": "val",
        "test": "test", "val": "val",
    }

    def __init__(
        self,
        split: str = "train",
        max_rir_len: int = DEFAULT_MAX_RIR_LEN,
        num_time_steps: int = 256,
        sample_rate: int = 16_000,
        cache_dir: Optional[str] = None,
    ) -> None:
        super().__init__()
        self.split = split
        self.max_rir_len = max_rir_len
        self.num_time_steps = num_time_steps
        self.sample_rate = sample_rate

        hf_split = "validation" if split == "val" else split

        # ---- 1. Load HF audio dataset -----------------------------------
        print(f"[RIRMegaDataset] Loading HF audio (split='{hf_split}') …")
        self._hf_ds = load_dataset(
            "mandipgoswami/rirmega",
            split=hf_split,
            cache_dir=cache_dir,
            token=HF_TOKEN if HF_TOKEN else None,
        )

        # ---- 2. Download & parse metadata CSV ---------------------------
        print("[RIRMegaDataset] Downloading metadata CSV …")
        meta_path = hf_hub_download(
            repo_id="mandipgoswami/rirmega",
            filename="metadata/metadata.csv",
            repo_type="dataset",
            cache_dir=cache_dir,
            token=HF_TOKEN if HF_TOKEN else None,
        )
        full_meta = pd.read_csv(meta_path)
        full_meta["id"] = full_meta["id"].astype(str).str.strip()

        # Filter metadata to requested split
        csv_split = self._HF_SPLIT_MAP.get(hf_split, split)
        self._meta = (
            full_meta[full_meta["split"].str.strip() == csv_split]
            .reset_index(drop=True)
        )
        self._meta_id_to_pos = {
            row["id"]: i for i, row in self._meta.iterrows()
        }

        # ---- 3. Build alignment (HF index <-> metadata row) ------------
        self._index_map = self._build_index()
        print(f"[RIRMegaDataset] Ready — {len(self)} aligned samples "
              f"(split='{split}')")

    # ------------------------------------------------------------------
    def _build_index(self) -> List[Tuple[int, int]]:
        """Align HF audio items with metadata rows by sample-ID.
        Falls back to positional matching when paths are unavailable.
        """
        raw = self._hf_ds.cast_column("audio", Audio(decode=False))
        paths = raw["audio"]  # list[dict]

        index_map: List[Tuple[int, int]] = []
        for hf_idx, item in enumerate(paths):
            path = item.get("path", "") if isinstance(item, dict) else ""
            sid = _extract_sample_id(path)
            if sid and sid in self._meta_id_to_pos:
                index_map.append((hf_idx, self._meta_id_to_pos[sid]))

        # Positional fallback
        if not index_map:
            n = min(len(self._hf_ds), len(self._meta))
            index_map = [(i, i) for i in range(n)]

        return index_map

    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self._index_map)

    # ------------------------------------------------------------------
    def _parse_meta_row(self, row: pd.Series) -> Dict:
        """Unpack all JSON-encoded fields for a single metadata row."""
        return {
            "room_size":        _parse_json_field(row["room_size"])       or [0., 0., 0.],
            "source":           _parse_json_field(row["source"])          or [0., 0., 0.],
            "microphone":       _parse_json_field(row["microphone"])      or [0., 0., 0.],
            "absorption":       float(row.get("absorption", 0.0)),
            "absorption_bands": _parse_json_field(row["absorption_bands"]) or {},
            "metrics":          _parse_json_field(row["metrics"])          or {},
        }

    # ------------------------------------------------------------------
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        hf_idx, meta_idx = self._index_map[idx]

        # ---- Audio (decoded on-the-fly by HF datasets) ----------------
        audio = self._hf_ds[hf_idx]["audio"]
        rir_np = np.asarray(audio["array"], dtype=np.float32)
        original_len = len(rir_np)

        # Pad / truncate to fixed length
        if len(rir_np) >= self.max_rir_len:
            rir_np = rir_np[: self.max_rir_len]
        else:
            rir_np = np.pad(rir_np, (0, self.max_rir_len - len(rir_np)))

        # Schroeder EDC (broadband)
        edc_np = compute_edc(rir_np)

        # Multiband EDC target [num_time_steps, num_bands]
        edc_mb_np = compute_multiband_edc(
            rir_np, sr=self.sample_rate,
            num_time_steps=self.num_time_steps,
        )

        # ---- Metadata -------------------------------------------------
        p = self._parse_meta_row(self._meta.iloc[meta_idx])

        band_abs = [float(p["absorption_bands"].get(b, 0.0))
                    for b in OCTAVE_BANDS]

        # Room acoustic mode features (computed from room dimensions)
        room_L, room_W, room_H = p["room_size"]
        modal_feats = compute_room_modes(room_L, room_W, room_H)

        x = torch.tensor(
            p["room_size"]                          # 3  room dims
            + p["source"]                           # 3  source xyz
            + p["microphone"]                       # 3  mic xyz
            + [p["absorption"]]                     # 1  broadband abs
            + band_abs                              # 6  band abs
            + modal_feats.tolist(),                  # 8  modal features
            dtype=torch.float32,
        )  # shape [24]

        m = p["metrics"]
        band_rt60 = m.get("band_rt60s", {})
        metrics_vec = (
            [float(m.get("rt60", 0.0)),
             float(m.get("drr_db", 0.0)),
             float(m.get("c50_db", 0.0)),
             float(m.get("c80_db", 0.0))]
            + [float(band_rt60.get(b, 0.0)) for b in OCTAVE_BANDS]
        )

        y = {
            "metrics":    torch.tensor(metrics_vec, dtype=torch.float32),   # [10]
            "rir":        torch.from_numpy(rir_np),                         # [max_rir_len]
            "edc":        torch.from_numpy(edc_np),                         # [max_rir_len]
            "edc_mb":     torch.from_numpy(edc_mb_np),                      # [num_time_steps, num_bands]
            "rir_length": torch.tensor(original_len, dtype=torch.long),     # scalar
        }

        return x, y

print("RIRMegaDataset class defined.")


In [ ]:
# ============================================================
# Cell 7 — Collate Function & DataLoader Factory
# ============================================================

def rir_collate_fn(batch):
    """Stack X tensors and collate Y dicts into batched tensors."""
    xs, ys = zip(*batch)
    return (
        torch.stack(xs, dim=0),
        {
            "metrics":    torch.stack([y["metrics"]    for y in ys], dim=0),
            "rir":        torch.stack([y["rir"]        for y in ys], dim=0),
            "edc":        torch.stack([y["edc"]        for y in ys], dim=0),
            "edc_mb":     torch.stack([y["edc_mb"]     for y in ys], dim=0),
            "rir_length": torch.stack([y["rir_length"] for y in ys], dim=0),
        },
    )


def get_dataloader(
    split: str = "train",
    batch_size: int = 32,
    max_rir_len: int = DEFAULT_MAX_RIR_LEN,
    num_time_steps: int = 256,
    sample_rate: int = 16_000,
    num_workers: int = 2,
    shuffle: Optional[bool] = None,
    cache_dir: Optional[str] = None,
    **loader_kwargs,
) -> DataLoader:
    """Convenience factory — returns a ready-to-iterate DataLoader."""
    if shuffle is None:
        shuffle = split == "train"

    ds = RIRMegaDataset(split=split, max_rir_len=max_rir_len,
                        num_time_steps=num_time_steps,
                        sample_rate=sample_rate,
                        cache_dir=cache_dir)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=rir_collate_fn,
        pin_memory=torch.cuda.is_available(),
        **loader_kwargs,
    )

print("Collate function & DataLoader factory defined.")

In [ ]:
# ============================================================
# Cell 8 — Unit Test: Verify Tensor Shapes
# ============================================================

MAX_LEN = DEFAULT_MAX_RIR_LEN
BATCH = 4

print("=" * 64)
print("  Unit Test — RIRMegaDataset  (Phase 1: Data Pipeline)")
print("=" * 64)

# ---- Single-sample test ----------------------------------------------
ds = RIRMegaDataset(split="train", max_rir_len=MAX_LEN)

x, y = ds[0]
print(f"\n[Single sample — index 0]")
print(f"  X shape      : {x.shape}           (expect [{INPUT_DIM}])")
print(f"  X dtype      : {x.dtype}")
print(f"  X values     : {x.tolist()[:6]} …")
print(f"  Y keys       : {sorted(y.keys())}")
print(f"  metrics shape: {y['metrics'].shape}  (expect [{METRICS_DIM}])")
print(f"  rir shape    : {y['rir'].shape}  (expect [{MAX_LEN}])")
print(f"  edc shape    : {y['edc'].shape}  (expect [{MAX_LEN}])")
print(f"  rir_length   : {y['rir_length'].item()}")

assert x.shape == (INPUT_DIM,),              f"X shape mismatch: {x.shape}"
assert y["metrics"].shape == (METRICS_DIM,), f"metrics shape mismatch: {y['metrics'].shape}"
assert y["rir"].shape == (MAX_LEN,),         f"rir shape mismatch: {y['rir'].shape}"
assert y["edc"].shape == (MAX_LEN,),         f"edc shape mismatch: {y['edc'].shape}"
assert y["rir_length"].dtype == torch.long

# ---- Batch / DataLoader test -----------------------------------------
loader = get_dataloader(
    split="train", batch_size=BATCH,
    max_rir_len=MAX_LEN, num_workers=0,
)
xb, yb = next(iter(loader))

print(f"\n[Batch — batch_size={BATCH}]")
print(f"  X batch      : {xb.shape}        (expect [{BATCH}, {INPUT_DIM}])")
print(f"  metrics batch: {yb['metrics'].shape}  (expect [{BATCH}, {METRICS_DIM}])")
print(f"  rir batch    : {yb['rir'].shape}  (expect [{BATCH}, {MAX_LEN}])")
print(f"  edc batch    : {yb['edc'].shape}  (expect [{BATCH}, {MAX_LEN}])")
print(f"  rir_lengths  : {yb['rir_length'].tolist()}")

assert xb.shape == (BATCH, INPUT_DIM)
assert yb["metrics"].shape == (BATCH, METRICS_DIM)
assert yb["rir"].shape == (BATCH, MAX_LEN)
assert yb["edc"].shape == (BATCH, MAX_LEN)

# ---- GPU transfer sanity check (Colab) --------------------------------
xb_gpu = xb.to(DEVICE)
print(f"\n  X on device  : {xb_gpu.device}")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — data pipeline is operational.")
print("=" * 64)

---
# Phase 2 — LSTM Multiband EDC Predictor

**Goal:** Map the 16-d physical input features → a predicted **multiband Energy Decay Curve** (dB) using an LSTM.

### Architecture

```
Input [B, 24]  (room dims, src/mic xyz, absorption)
       │
  ┌────▼────┐
  │   MLP   │  2-layer encoder
  │ Encoder │  → context vector [B, hidden]
  └────┬────┘
       │  projected into h₀, c₀ for each LSTM layer
  ┌────▼────────────────────────┐
  │  LSTM  (T learned temporal  │
  │         input embeddings)   │  num_layers, hidden_dim
  └────┬────────────────────────┘
       │  [B, T, hidden]
  ┌────▼────┐
  │ Linear  │  projection head
  │  Head   │  → [B, T, num_bands]  (EDC in dB)
  └─────────┘
```

| Param | Default | Meaning |
|-------|---------|---------|
| `input_dim` | 16 | Physical features from dataset X |
| `hidden_dim` | 256 | LSTM & encoder width |
| `num_layers` | 2 | Stacked LSTM layers |
| `num_bands` | 6 | Octave bands (125 – 4000 Hz) |
| `num_time_steps` | 256 | EDC time-frames (~7.8 ms each @ 2 s) |

In [ ]:
# ============================================================
# Cell 10 — LSTM Multiband EDC Predictor
# ============================================================
import torch.nn as nn


class MultibandEDCPredictor(nn.Module):
    """LSTM-based predictor: physical room parameters -> multiband EDC.

    The model conditions an LSTM on the room/source/absorption features
    via learned initial hidden & cell states, then unrolls over *T*
    time-steps using a learned temporal embedding to produce a
    frequency-dependent Energy Decay Curve.

    Parameters
    ----------
    input_dim : int
        Dimensionality of physics input vector (default 24 = 16 base + 8 modal).
    hidden_dim : int
        Width of the encoder MLP and LSTM hidden state.
    num_layers : int
        Number of stacked LSTM layers.
    num_bands : int
        Number of output frequency bands (default 6 octave bands).
    num_time_steps : int
        Temporal resolution of the predicted EDC.
    dropout : float
        Dropout applied between LSTM layers (ignored when num_layers=1).
    """

    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_bands: int = len(OCTAVE_BANDS),     # 6
        num_time_steps: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_bands = num_bands
        self.num_time_steps = num_time_steps

        # ---- Input normalisation ----------------------------------------
        #   Crucial: inputs span wildly different scales (absorption ~0.1,
        #   room dims ~1-10 m, modal freqs ~30-2000 Hz).  LayerNorm normalises
        #   over the feature dimension and works for any batch size (incl. B=1).
        self.input_norm = nn.LayerNorm(input_dim)

        # ---- MLP encoder: input features -> context vector ---------------
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
        )

        # ---- Project context -> LSTM initial states ----------------------
        #   h0 and c0 each have shape [num_layers, B, hidden_dim]
        self.h0_proj = nn.Linear(hidden_dim, num_layers * hidden_dim)
        self.c0_proj = nn.Linear(hidden_dim, num_layers * hidden_dim)

        # ---- Learned temporal input embeddings --------------------------
        #   One embedding vector per time-step, shared across the batch
        self.time_embed = nn.Parameter(
            torch.randn(1, num_time_steps, hidden_dim) * 0.02
        )

        # ---- LSTM -------------------------------------------------------
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # ---- Output projection: hidden → log-decrements → cumsum → monotone EDC
        #   Two-layer head with enhanced capacity; cumsum in forward() ensures
        #   the predicted dB EDC is always monotonically decreasing.
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, num_bands),
        )

        self._init_weights()

    # ------------------------------------------------------------------
    def _init_weights(self) -> None:
        """Xavier init for linear layers; orthogonal for LSTM weights."""
        for name, param in self.named_parameters():
            if "lstm" in name and "weight" in name:
                nn.init.orthogonal_(param)
            elif "lstm" in name and "bias" in name:
                nn.init.zeros_(param)
                # Set forget-gate bias to 1 for better gradient flow
                hidden = self.hidden_dim
                param.data[hidden : 2 * hidden].fill_(1.0)
            elif param.dim() >= 2 and "time_embed" not in name:
                nn.init.xavier_uniform_(param)

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor[B, input_dim]
            Physical input features from the dataset.

        Returns
        -------
        edc_pred : Tensor[B, num_time_steps, num_bands]
            Predicted multiband Energy Decay Curve (dB scale).
        """
        B = x.size(0)

        # 1. Normalise & encode input features
        # LayerNorm normalises over the last dim — works for any batch size
        x = self.input_norm(x)                         # [B, input_dim]
        ctx = self.encoder(x)                          # [B, hidden]

        # 2. Derive LSTM initial states from context
        h0 = (
            self.h0_proj(ctx)                          # [B, num_layers*hidden]
            .view(B, self.num_layers, self.hidden_dim)
            .permute(1, 0, 2)                          # [num_layers, B, hidden]
            .contiguous()
        )
        c0 = (
            self.c0_proj(ctx)
            .view(B, self.num_layers, self.hidden_dim)
            .permute(1, 0, 2)
            .contiguous()
        )

        # 3. Expand temporal embeddings to batch -> LSTM input
        lstm_in = self.time_embed.expand(B, -1, -1)   # [B, T, hidden]

        # 4. LSTM forward pass
        lstm_out, _ = self.lstm(lstm_in, (h0, c0))    # [B, T, hidden]

        # 5. Project to log-decrements, then cumsum for monotonic dB EDC.
        #    Negative cumsum ensures EDC decreases over time (as a real decay).
        log_dec = self.output_head(lstm_out)            # [B, T, num_bands]
        # Softplus ensures non-negative decrements (decay must go down)
        decrements = torch.nn.functional.softplus(log_dec) * 0.5
        # Cumulative sum then negate → 0, -d1, -(d1+d2), ... (monotone decrease)
        edc_pred = -torch.cumsum(decrements, dim=1)     # [B, T, num_bands]

        return edc_pred

    # ------------------------------------------------------------------
    def predict_broadband(self, x: torch.Tensor) -> torch.Tensor:
        """Convenience: average predicted band EDCs -> single broadband EDC.

        Returns
        -------
        edc_broadband : Tensor[B, num_time_steps]
        """
        return self.forward(x).mean(dim=-1)

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("MultibandEDCPredictor class defined (with BatchNorm1d input normalisation).")

In [ ]:
# ============================================================
# Cell 11 — Unit Test: LSTM Model Shape Verification
# ============================================================

BATCH_TEST = 4
NUM_BANDS = len(OCTAVE_BANDS)   # 6
T_STEPS = 256

print("=" * 64)
print("  Unit Test — MultibandEDCPredictor  (Phase 2: LSTM Model)")
print("=" * 64)

# ---- Instantiate model ------------------------------------------------
model = MultibandEDCPredictor(
    input_dim=INPUT_DIM,
    hidden_dim=256,
    num_layers=2,
    num_bands=NUM_BANDS,
    num_time_steps=T_STEPS,
    dropout=0.1,
).to(DEVICE)

print(f"\n  Model parameters : {model.count_parameters():,}")
print(f"  Device           : {DEVICE}")

# ---- Forward pass with dummy data ------------------------------------
dummy_x = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE)
with torch.no_grad():
    edc_pred = model(dummy_x)

print(f"\n[Forward pass — dummy batch]")
print(f"  Input  X shape   : {dummy_x.shape}       (expect [{BATCH_TEST}, {INPUT_DIM}])")
print(f"  Output EDC shape : {edc_pred.shape}  (expect [{BATCH_TEST}, {T_STEPS}, {NUM_BANDS}])")
print(f"  Output dtype     : {edc_pred.dtype}")
print(f"  Output range     : [{edc_pred.min().item():.4f}, {edc_pred.max().item():.4f}]")

assert edc_pred.shape == (BATCH_TEST, T_STEPS, NUM_BANDS), \
    f"Shape mismatch: {edc_pred.shape}"
assert edc_pred.device.type == DEVICE.type

# ---- Broadband convenience method ------------------------------------
with torch.no_grad():
    edc_broad = model.predict_broadband(dummy_x)

print(f"\n[Broadband EDC]")
print(f"  Shape            : {edc_broad.shape}      (expect [{BATCH_TEST}, {T_STEPS}])")

assert edc_broad.shape == (BATCH_TEST, T_STEPS)

# ---- Gradient flow check ---------------------------------------------
model.train()
x_grad = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE, requires_grad=True)
pred = model(x_grad)
loss = pred.sum()
loss.backward()

grad_ok = x_grad.grad is not None and x_grad.grad.abs().sum().item() > 0
print(f"\n  Gradients flow   : {'✓ OK' if grad_ok else '✗ BROKEN'}")
assert grad_ok, "Gradients do not flow back to input!"

# ---- Integration with Phase 1 dataset (if loaded) --------------------
try:
    sample_x, _ = ds[0]               # from Phase 1 unit test
    # Guard: if dataset was built before modal features, skip stale sample
    if sample_x.shape[0] != INPUT_DIM:
        print(f"\n  ⚠ Dataset sample dim ({sample_x.shape[0]}) != INPUT_DIM ({INPUT_DIM})")
        print("    Re-run Phase 1 (cell 7) to rebuild the dataset with modal features.")
    else:
        sample_x = sample_x.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            real_pred = model(sample_x)
        print(f"\n[Real sample from dataset]")
        print(f"  Input  X         : {sample_x.shape}")
        print(f"  Output EDC       : {real_pred.shape}")
        print(f"  EDC[0, 0, :]     : {real_pred[0, 0, :].tolist()}")  # first time-step, all bands
except NameError:
    print("\n  (Skipping real-sample test — run Phase 1 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — LSTM model is operational.")
print("=" * 64)

---
# Phase 3 — Custom Loss Functions (Physics-Informed)

**Goal:** Combine a standard EDC reconstruction loss with physics-informed regularization terms derived from the fundamental equations of linear room acoustics.

---

### 1. EDC Reconstruction Loss

Standard root-mean-square error between predicted and target multiband EDC:

$$\mathcal{L}_{\text{EDC}} = \sqrt{\frac{1}{B \cdot T \cdot F}\sum_{b,t,f}\bigl(\hat{E}_{b,t,f} - E_{b,t,f}\bigr)^2}$$

---

### 2. Acoustic Continuity Equation (Mass Conservation)

In lossless linear acoustics the continuity equation relates pressure $p$ and particle velocity $\mathbf{u}$:

$$\frac{\partial p}{\partial t} + \rho_0 c^2 \nabla \cdot \mathbf{u} = 0$$

We penalise the residual so the network learns spatially consistent fields:

$$\mathcal{L}_{\text{cont}} = \frac{1}{N}\sum_i\left(\frac{\partial p_i}{\partial t} + \rho_0 c^2 \left(\frac{\partial u_x}{\partial x} + \frac{\partial u_y}{\partial y} + \frac{\partial u_z}{\partial z}\right)_i\right)^2$$

---

### 3. Linearized Momentum Equation (Euler's Equation)

Newton's second law for an inviscid fluid:

$$\rho_0 \frac{\partial \mathbf{u}}{\partial t} + \nabla p = 0$$

Residual computed **per spatial component**:

$$\mathcal{L}_{\text{mom}} = \frac{1}{N}\sum_i\left\|\rho_0\frac{\partial \mathbf{u}_i}{\partial t} + \nabla p_i\right\|^2$$

---

### Combined Loss

$$\mathcal{L} = \mathcal{L}_{\text{EDC}} + \lambda_{\text{cont}}\,\mathcal{L}_{\text{cont}} + \lambda_{\text{mom}}\,\mathcal{L}_{\text{mom}}$$

| Hyper-param | Default | Meaning |
|-------------|---------|---------|
| `lambda_cont` | 0.01 | Weight for continuity residual |
| `lambda_mom` | 0.01 | Weight for momentum residual |
| `rho0` | 1.225 | Air density (kg/m³) |
| `c` | 343.0 | Speed of sound (m/s) |

In [ ]:
# ============================================================
# Cell 13 — Physics-Informed Loss Functions
# ============================================================
import torch
import torch.nn as nn
from typing import Dict, Optional


# ------------------------------------------------------------------
#  1.  EDC Reconstruction Loss (RMSE)
# ------------------------------------------------------------------

class EDCReconstructionLoss(nn.Module):
    """Weighted RMSE between predicted and target EDC (dB).

    The standard RMSE over 0..-60 dB treats late-decay errors equally to
    early-decay errors, but early decay structure (first 10 dB) is
    perceptually and physically more important.  This loss adds:
      1. Early-emphasis weighting (higher weight in first 25% of time steps)
      2. Slope-matching penalty that encourages the correct RT60 gradient
      3. Standard RMSE term

    Supports both multiband [B, T, F] and broadband [B, T] tensors.
    """

    def __init__(
        self,
        early_weight: float = 3.0,
        slope_weight: float = 0.5,
        decay_rate: float = 5.0,
    ) -> None:
        """
        Parameters
        ----------
        early_weight : float
            Multiplier applied to the first time step (weight decays to 1.0).
        slope_weight : float
            Weight applied to the finite-difference slope-matching penalty.
        decay_rate : float
            Controls how quickly the early-emphasis weight decays toward 1.
            A value of 5.0 ensures ~40% of time steps receive elevated emphasis.
        """
        super().__init__()
        self.early_weight = early_weight
        self.slope_weight = slope_weight
        self.decay_rate = decay_rate

    def forward(
        self,
        edc_pred: torch.Tensor,
        edc_target: torch.Tensor,
    ) -> torch.Tensor:
        # edc_pred, edc_target: [B, T, F] or [B, T]
        T = edc_pred.shape[1]
        # Build early-emphasis weight vector: 3× for first 25%, tapering to 1×
        t = torch.arange(T, device=edc_pred.device, dtype=edc_pred.dtype) / T
        w = 1.0 + (self.early_weight - 1.0) * torch.exp(-self.decay_rate * t)   # [T]
        # Broadcast over batch / band dimensions
        while w.dim() < edc_pred.dim():
            w = w.unsqueeze(-1)
        w = w.permute(*([0] + list(range(1, w.dim()))))              # [T, ...]
        # Reshape to match edc dimensions
        w = w.squeeze(-1) if edc_pred.dim() == 2 else w
        # For [B, T] or [B, T, F] we need w to be broadcastable
        if edc_pred.dim() == 3:
            w_broadcast = w.view(1, T, 1)
        else:
            w_broadcast = w.view(1, T)

        # 1. Weighted RMSE
        diff = (edc_pred - edc_target) * w_broadcast
        rmse = torch.sqrt(torch.mean(diff ** 2) + 1e-8)

        # 2. Slope penalty: penalise difference in EDC gradient (~ RT60 proxy)
        slope_pred = edc_pred[:, 1:] - edc_pred[:, :-1]    # finite diff
        slope_tgt  = edc_target[:, 1:] - edc_target[:, :-1]
        slope_loss = torch.sqrt(torch.mean((slope_pred - slope_tgt) ** 2) + 1e-8)

        return rmse + self.slope_weight * slope_loss


# ------------------------------------------------------------------
#  2.  Acoustic Continuity Equation Residual
# ------------------------------------------------------------------

def continuity_residual(
    pressure: torch.Tensor,
    velocity: torch.Tensor,
    coords: torch.Tensor,
    time: torch.Tensor,
    rho0: float = 1.225,
    c: float = 343.0,
) -> torch.Tensor:
    """Residual of the acoustic continuity equation.

    ∂p/∂t  +  ρ₀ c² ∇·u  =  0

    All inputs must have `requires_grad=True` so that
    `torch.autograd.grad` can compute the necessary derivatives.

    Parameters
    ----------
    pressure : Tensor[N, 1]
        Acoustic pressure field at N collocation points.
    velocity : Tensor[N, 3]
        Particle velocity (u_x, u_y, u_z) at the same points.
    coords : Tensor[N, 3]
        Spatial coordinates (x, y, z) of each collocation point.
        Must have `requires_grad=True`.
    time : Tensor[N, 1]
        Time coordinate for each point.
        Must have `requires_grad=True`.
    rho0 : float
        Equilibrium air density (kg/m³).
    c : float
        Speed of sound (m/s).

    Returns
    -------
    residual : Tensor[N, 1]
        Point-wise residual of the continuity equation.
    """
    # ∂p/∂t  — temporal derivative of pressure
    dp_dt = torch.autograd.grad(
        outputs=pressure,
        inputs=time,
        grad_outputs=torch.ones_like(pressure),
        create_graph=True,
        retain_graph=True,
        allow_unused=True,
    )[0]                                                # [N, 1]
    if dp_dt is None:
        dp_dt = torch.zeros_like(pressure)

    # ∂u_x/∂x, ∂u_y/∂y, ∂u_z/∂z  — spatial divergence of velocity
    div_u = torch.zeros_like(pressure)                  # [N, 1]
    for dim in range(3):
        du_i_full = torch.autograd.grad(
            outputs=velocity[:, dim : dim + 1],
            inputs=coords,
            grad_outputs=torch.ones_like(pressure),
            create_graph=True,
            retain_graph=True,
            allow_unused=True,
        )[0]                                            # [N, 3] or None
        if du_i_full is not None:
            div_u = div_u + du_i_full[:, dim : dim + 1] # ∂u_i/∂x_i

    return dp_dt + rho0 * c ** 2 * div_u                # [N, 1]


# ------------------------------------------------------------------
#  3.  Linearized Momentum Equation Residual (Euler's Equation)
# ------------------------------------------------------------------

def momentum_residual(
    pressure: torch.Tensor,
    velocity: torch.Tensor,
    coords: torch.Tensor,
    time: torch.Tensor,
    rho0: float = 1.225,
) -> torch.Tensor:
    """Residual of the linearized momentum equation.

    ρ₀ ∂u/∂t  +  ∇p  =  0

    Parameters
    ----------
    pressure : Tensor[N, 1]
        Acoustic pressure at N collocation points.
    velocity : Tensor[N, 3]
        Particle velocity (u_x, u_y, u_z).
    coords : Tensor[N, 3]
        Spatial coordinates (requires_grad=True).
    time : Tensor[N, 1]
        Temporal coordinate (requires_grad=True).
    rho0 : float
        Equilibrium air density.

    Returns
    -------
    residual : Tensor[N, 3]
        Per-component residual of the momentum equation.
    """
    # ∇p  — spatial gradient of pressure  [N, 3]
    grad_p = torch.autograd.grad(
        outputs=pressure,
        inputs=coords,
        grad_outputs=torch.ones_like(pressure),
        create_graph=True,
        retain_graph=True,
        allow_unused=True,
    )[0]                                                # [N, 3] or None
    if grad_p is None:
        grad_p = torch.zeros(pressure.size(0), 3, device=pressure.device)

    # ∂u/∂t  — temporal derivative of each velocity component  [N, 3]
    du_dt = torch.autograd.grad(
        outputs=velocity,
        inputs=time,
        grad_outputs=torch.ones_like(velocity),
        create_graph=True,
        retain_graph=True,
        allow_unused=True,
    )[0]                                                # [N, 3] or None
    if du_dt is None:
        du_dt = torch.zeros_like(velocity)

    return rho0 * du_dt + grad_p                        # [N, 3]


# ------------------------------------------------------------------
#  4.  Combined Physics-Informed Loss
# ------------------------------------------------------------------

class PhysicsInformedRIRLoss(nn.Module):
    """Unified loss = L_edc  +  λ_cont · L_cont  +  λ_mom · L_mom.

    The EDC term is always computed.  The physics residual terms are
    only added when the caller supplies the collocation-point fields
    (pressure, velocity, coords, time).  This makes the loss usable
    both for **pure EDC regression** (early training) and for
    **physics-regularised fine-tuning** (later phases).

    Parameters
    ----------
    lambda_cont : float
        Weight for the continuity-equation residual.
    lambda_mom : float
        Weight for the momentum-equation residual.
    rho0 : float
        Equilibrium air density (kg/m³).
    c : float
        Speed of sound (m/s).
    """

    def __init__(
        self,
        lambda_cont: float = 0.01,
        lambda_mom: float = 0.01,
        rho0: float = 1.225,
        c: float = 343.0,
    ) -> None:
        super().__init__()
        self.lambda_cont = lambda_cont
        self.lambda_mom = lambda_mom
        self.rho0 = rho0
        self.c = c
        self.edc_loss_fn = EDCReconstructionLoss(early_weight=3.0, slope_weight=0.5)

    def forward(
        self,
        edc_pred: torch.Tensor,
        edc_target: torch.Tensor,
        pressure: Optional[torch.Tensor] = None,
        velocity: Optional[torch.Tensor] = None,
        coords: Optional[torch.Tensor] = None,
        time: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Parameters
        ----------
        edc_pred : Tensor[B, T, F] or [B, T]
            Predicted EDC from the LSTM model.
        edc_target : Tensor — same shape as edc_pred
            Ground-truth EDC from the dataset.
        pressure : Tensor[N, 1], optional
            Acoustic pressure at N collocation points.
        velocity : Tensor[N, 3], optional
            Particle velocity at the same points.
        coords : Tensor[N, 3], optional  (requires_grad=True)
            Spatial coordinates of collocation points.
        time : Tensor[N, 1], optional    (requires_grad=True)
            Temporal coordinates of collocation points.

        Returns
        -------
        losses : dict
            "total"     — combined scalar loss (for .backward())
            "edc"       — EDC RMSE component
            "continuity"— continuity residual MSE  (0 if not provided)
            "momentum"  — momentum residual MSE    (0 if not provided)
        """
        device = edc_pred.device

        # ---- 1. EDC reconstruction loss --------------------------------
        l_edc = self.edc_loss_fn(edc_pred, edc_target)

        # ---- 2. Physics residuals (optional) ---------------------------
        has_physics = all(
            t is not None for t in [pressure, velocity, coords, time]
        )

        if has_physics:
            # Continuity: ∂p/∂t + ρ₀c² ∇·u = 0
            r_cont = continuity_residual(
                pressure, velocity, coords, time,
                rho0=self.rho0, c=self.c,
            )
            l_cont = torch.mean(r_cont ** 2)

            # Momentum: ρ₀ ∂u/∂t + ∇p = 0
            r_mom = momentum_residual(
                pressure, velocity, coords, time,
                rho0=self.rho0,
            )
            l_mom = torch.mean(r_mom ** 2)
        else:
            l_cont = torch.tensor(0.0, device=device)
            l_mom = torch.tensor(0.0, device=device)

        # ---- 3. Combine ------------------------------------------------
        total = l_edc + self.lambda_cont * l_cont + self.lambda_mom * l_mom

        return {
            "total": total,
            "edc": l_edc,
            "continuity": l_cont,
            "momentum": l_mom,
        }




# ======================================================================
#  5.  Multi-Resolution STFT Loss
# ======================================================================

def _stft_mag_phase(x, fft_size, hop_length, win_length):
    """Compute STFT magnitude and rectangular-coordinate phase."""
    window = torch.hann_window(win_length, device=x.device, dtype=x.dtype)
    stft = torch.stft(x, fft_size, hop_length, win_length, window=window, return_complex=True)
    mag = stft.abs()
    phase_cos = stft.real / (mag + 1e-8)
    phase_sin = stft.imag / (mag + 1e-8)
    return mag, phase_cos, phase_sin


class MultiResolutionSTFTLoss(nn.Module):
    """Multi-Resolution STFT loss over several window lengths.

    Combines spectral magnitude (L1), log-magnitude MSE, and
    rectangular-coordinate phase loss to penalise the generator over
    multiple time-frequency resolutions simultaneously.
    """

    def __init__(self, window_lengths=None):
        super().__init__()
        self.window_lengths = window_lengths or [512, 1024, 2048]

    def forward(self, pred, target):
        """
        Parameters
        ----------
        pred, target : Tensor[B, T]  time-domain waveforms
        """
        import torch.nn.functional as F
        loss = torch.zeros((), device=pred.device, dtype=pred.dtype)
        for wl in self.window_lengths:
            hop = wl // 4
            mag_p, cos_p, sin_p = _stft_mag_phase(pred, wl, hop, wl)
            mag_t, cos_t, sin_t = _stft_mag_phase(target, wl, hop, wl)
            loss = loss + F.l1_loss(mag_p, mag_t)
            loss = loss + F.mse_loss(
                torch.log(mag_p + 1e-7), torch.log(mag_t + 1e-7)
            )
            loss = loss + F.mse_loss(cos_p, cos_t) + F.mse_loss(sin_p, sin_t)
        return loss / len(self.window_lengths)


# ======================================================================
#  6.  Collocation Physics Loss  (proper PINN formulation)
# ======================================================================

class CollocationPhysicsLoss(nn.Module):
    """Physics loss computed over randomly sampled (x, y, z, t) points.

    Uses a SIREN coordinate network (SIRENCoordinateNet, defined in
    the models cell) to predict pressure p and velocity (ux, uy, uz)
    at random interior points, then enforces the acoustic wave equations
    via ``torch.autograd.grad``.  This is the proper PINN formulation
    that forces the network to obey wave propagation laws.

    Parameters
    ----------
    coord_net : nn.Module
        Network mapping [N, 4] -> [N, 4] where outputs are (p, ux, uy, uz).
    lambda_cont, lambda_mom : float
        Weights for the continuity and momentum residuals.
    rho0 : float
        Equilibrium air density (kg/m³).
    c : float
        Speed of sound (m/s).
    """

    def __init__(self, coord_net, lambda_cont=0.01, lambda_mom=0.01,
                 rho0=1.225, c=343.0):
        super().__init__()
        self.coord_net = coord_net
        self.lambda_cont = lambda_cont
        self.lambda_mom = lambda_mom
        self.rho0 = rho0
        self.c = c

    def forward(self, room_dims, n_points=128):
        """Sample random collocation points and compute physics loss.

        Parameters
        ----------
        room_dims : Tensor[B, 3]
            Room dimensions (L, W, H) in metres.
        n_points : int
            Number of collocation points to sample.
        """
        device = room_dims.device
        dtype = room_dims.dtype
        room_max = room_dims.mean(dim=0).clamp(min=0.1)    # [3]

        coords = torch.rand(n_points, 3, device=device, dtype=dtype,
                            requires_grad=True)
        coords_scaled = coords * room_max.unsqueeze(0)      # metres
        time = torch.rand(n_points, 1, device=device, dtype=dtype,
                          requires_grad=True) * 2.0         # [0, 2] s

        xyzT = torch.cat([coords_scaled, time], dim=1)     # [N, 4]
        pv = self.coord_net(xyzT)                           # [N, 4]
        pressure = pv[:, :1]                               # [N, 1]
        velocity = pv[:, 1:]                               # [N, 3]

        loss = torch.zeros((), device=device, dtype=dtype)
        if self.lambda_cont != 0.0:
            r_cont = continuity_residual(
                pressure, velocity, coords_scaled, time,
                rho0=self.rho0, c=self.c,
            )
            loss = loss + self.lambda_cont * torch.mean(r_cont ** 2)
        if self.lambda_mom != 0.0:
            r_mom = momentum_residual(
                pressure, velocity, coords_scaled, time,
                rho0=self.rho0,
            )
            loss = loss + self.lambda_mom * torch.mean(r_mom ** 2)
        return loss


print("MultiResolutionSTFTLoss and CollocationPhysicsLoss defined.")
print("PhysicsInformedRIRLoss and residual functions defined.")


In [ ]:
# ============================================================
# Cell 14 — Unit Test: Physics-Informed Loss Functions
# ============================================================

BATCH_TEST = 4
T_STEPS = 256
NUM_BANDS = len(OCTAVE_BANDS)   # 6
N_COLLOC = 128                   # collocation points for PDE test

print("=" * 64)
print("  Unit Test — PhysicsInformedRIRLoss  (Phase 3: Loss Functions)")
print("=" * 64)

# ---- 1. Pure EDC loss (no physics terms) -----------------------------
criterion = PhysicsInformedRIRLoss(
    lambda_cont=0.01, lambda_mom=0.01,
    rho0=1.225, c=343.0,
).to(DEVICE)

edc_pred   = torch.randn(BATCH_TEST, T_STEPS, NUM_BANDS, device=DEVICE)
edc_target = torch.randn(BATCH_TEST, T_STEPS, NUM_BANDS, device=DEVICE)

losses = criterion(edc_pred, edc_target)

print(f"\n[EDC-only mode — no physics terms]")
print(f"  total        : {losses['total'].item():.6f}")
print(f"  edc (RMSE)   : {losses['edc'].item():.6f}")
print(f"  continuity   : {losses['continuity'].item():.6f}  (expect 0)")
print(f"  momentum     : {losses['momentum'].item():.6f}  (expect 0)")

assert losses["continuity"].item() == 0.0, "Continuity should be 0 when not supplied"
assert losses["momentum"].item()   == 0.0, "Momentum should be 0 when not supplied"
assert losses["total"].item() > 0.0,       "Total loss should be > 0"
assert torch.isclose(losses["total"], losses["edc"]), \
    "Total should equal EDC when physics terms are absent"

# ---- 2. Standalone continuity residual test --------------------------
print(f"\n[Continuity residual — synthetic collocation points]")

coords = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)

# Build synthetic P(x,y,z,t) and u(x,y,z,t) as differentiable
# functions of coords & time so autograd can compute derivatives.
#   p = sin(x + y + z + t)   →  dp/dt = cos(...)
#   u = [sin(t), sin(t), sin(t)]  →  ∂u_x/∂x=0, etc, du/dt = cos(t)
pressure = torch.sin(coords.sum(dim=1, keepdim=True) + time)  # [N,1]
velocity = torch.cat([torch.sin(time)] * 3, dim=1)            # [N,3]

r_cont = continuity_residual(pressure, velocity, coords, time)
print(f"  Residual shape : {r_cont.shape}  (expect [{N_COLLOC}, 1])")
print(f"  Residual range : [{r_cont.min().item():.4f}, {r_cont.max().item():.4f}]")
assert r_cont.shape == (N_COLLOC, 1)

# Gradient should flow back
l_cont = torch.mean(r_cont ** 2)
l_cont.backward(retain_graph=True)
assert coords.grad is not None, "Gradients must flow to coords"
print(f"  Grad flows ✓")

# ---- 3. Standalone momentum residual test ----------------------------
print(f"\n[Momentum residual — synthetic collocation points]")

coords2 = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time2   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)

pressure2 = torch.sin(coords2.sum(dim=1, keepdim=True) + time2)
velocity2 = torch.cat([torch.sin(time2)] * 3, dim=1)

r_mom = momentum_residual(pressure2, velocity2, coords2, time2)
print(f"  Residual shape : {r_mom.shape}  (expect [{N_COLLOC}, 3])")
print(f"  Residual range : [{r_mom.min().item():.4f}, {r_mom.max().item():.4f}]")
assert r_mom.shape == (N_COLLOC, 3)

l_mom = torch.mean(r_mom ** 2)
l_mom.backward(retain_graph=True)
assert coords2.grad is not None, "Gradients must flow to coords"
print(f"  Grad flows ✓")

# ---- 4. Full combined loss with physics terms ------------------------
print(f"\n[Full physics-informed loss]")

coords3 = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time3   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)
p3 = torch.sin(coords3.sum(dim=1, keepdim=True) + time3)
v3 = torch.cat([torch.sin(time3)] * 3, dim=1)

full_losses = criterion(
    edc_pred, edc_target,
    pressure=p3, velocity=v3, coords=coords3, time=time3,
)

print(f"  total        : {full_losses['total'].item():.6f}")
print(f"  edc (RMSE)   : {full_losses['edc'].item():.6f}")
print(f"  continuity   : {full_losses['continuity'].item():.6f}")
print(f"  momentum     : {full_losses['momentum'].item():.6f}")

assert full_losses["total"] > full_losses["edc"], \
    "Total should exceed pure EDC when physics terms are active"
assert full_losses["continuity"].item() > 0
assert full_losses["momentum"].item() > 0

# Backward through the full combined loss
full_losses["total"].backward()
assert coords3.grad is not None
print(f"  Full backward ✓")

# ---- 5. Integration with Phase 2 model (if loaded) ------------------
try:
    model.eval()
    dummy_x = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE)
    with torch.no_grad():
        pred = model(dummy_x)
    # Use random targets to simulate ground-truth
    target = torch.randn_like(pred)
    integration_losses = criterion(pred, target)
    print(f"\n[Integration: LSTM output → Loss]")
    print(f"  pred shape   : {pred.shape}")
    print(f"  EDC loss     : {integration_losses['edc'].item():.6f}")
except NameError:
    print("\n  (Skipping integration — run Phase 2 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — loss functions are operational.")
print("=" * 64)

---
# Phase 4 — Differentiable Feedback Delay Network (FDN)

**Goal:** Model the late reverberation tail with a 16-channel Feedback Delay Network whose parameters are **fully differentiable**, enabling end-to-end gradient-based optimisation.

---

### FDN Signal Flow

```
input x[n] ──► [b_0 ... b_15] gain ──┐
                                      │
              ┌───────────────────────▼───────────────────────┐
              │        16 parallel delay lines                │
              │   z^{-m_0}, z^{-m_1}, ..., z^{-m_15}         │
              │        (learnable log-spaced delays)          │
              └───────────────┬───────────────────────────────┘
                              │
                       ┌──────▼──────┐
                       │  16x16      │
                       │ Feedback    │ ◄── Hadamard / householder
                       │  Matrix A   │     (unitary, lossless)
                       └──────┬──────┘
                              │
              ┌───────────────▼───────────────────────────────┐
              │   per-channel absorption filter                │
              │   g_i = alpha_i · exp(-beta_i · m_i / fs)     │
              │        (frequency-independent decay)           │
              └───────────────┬───────────────────────────────┘
                              │
                       feedback ──► back to delay input
                              │
              ┌───────────────▼──────┐
              │  [c_0 ... c_15] gain │ ──► output y[n]
              └──────────────────────┘
```

### Learnable Parameters

| Parameter | Shape | Constraint | Meaning |
|-----------|:-----:|:-----------|:--------|
| `log_delays` | `[16]` | Sigmoid → [0, 50 ms] | Delay-line lengths in log-space |
| `alpha` | `[16]` | Sigmoid → (0, 1) | Per-channel decay rate |
| `beta` | `[16]` | Softplus → > 0 | Per-channel absorption scale |
| `input_gains` | `[16]` | Unconstrained | Input bus gains $b_i$ |
| `output_gains` | `[16]` | Unconstrained | Output bus gains $c_i$ |

In [ ]:
# ============================================================
# Cell 16 — Differentiable Feedback Delay Network (FDN)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional


# ======================================================================
#  SIREN Layer & Coordinate Network
# ======================================================================

class SirenLayer(nn.Module):
    """Sinusoidal representation layer (SIREN) with sine activation.

    Uses ``sin(omega_0 * linear(x))`` to ensure smooth, infinitely
    differentiable outputs — critical for physics residuals.
    """

    def __init__(self, in_features, out_features, omega_0=30.0, is_first=False):
        super().__init__()
        self.omega_0 = omega_0
        self.linear = nn.Linear(in_features, out_features)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1.0 / in_features, 1.0 / in_features)
            else:
                bound = (6.0 / in_features) ** 0.5 / omega_0
                self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SIRENCoordinateNet(nn.Module):
    """SIREN MLP that maps (x, y, z, t) coordinates -> (p, u_x, u_y, u_z).

    Used as the coordinate network in collocation-based PINN training.
    All layers use sinusoidal activation — essential for
    ``torch.autograd.grad``-based physics residuals.

    Parameters
    ----------
    hidden_dim : int
        Width of each hidden SIREN layer.
    num_layers : int
        Number of hidden layers (>= 1).
    omega_0 : float
        Frequency multiplier for the first layer.
    """

    def __init__(self, hidden_dim=64, num_layers=3, omega_0=30.0):
        super().__init__()
        layers = [SirenLayer(4, hidden_dim, omega_0=omega_0, is_first=True)]
        for _ in range(num_layers - 1):
            layers.append(SirenLayer(hidden_dim, hidden_dim, omega_0=omega_0))
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(hidden_dim, 4)               # p, ux, uy, uz

    def forward(self, xyzT):
        return self.out(self.net(xyzT))


# ======================================================================
#  Feedback Delay Network (FDN)
# ======================================================================

def _hadamard_matrix(n: int) -> torch.Tensor:
    """Construct a normalised Hadamard matrix of order *n* (must be power of 2).

    The Hadamard matrix is orthogonal / unitary, guaranteeing lossless
    energy circulation in the feedback loop.
    """
    assert n > 0 and (n & (n - 1)) == 0, "n must be a power of 2"
    H = torch.tensor([[1.0]])
    while H.size(0) < n:
        H = torch.cat([
            torch.cat([H,  H], dim=1),
            torch.cat([H, -H], dim=1),
        ], dim=0)
    return H / math.sqrt(n)


class DifferentiableFDN(nn.Module):
    """Feedback Delay Network with fully differentiable parameters.

    The module synthesises the **late reverberation tail** of a Room
    Impulse Response by circulating energy through *N* parallel delay
    lines connected by a unitary feedback matrix.

    All timing and decay parameters live in unconstrained spaces and are
    mapped to their physical ranges via smooth, differentiable
    activations so that standard gradient-based optimisers can tune them.

    Parameters
    ----------
    num_delays : int
        Number of parallel feedback loops (default 16).
    max_delay_ms : float
        Upper bound for delay-line lengths in milliseconds (default 50).
    sample_rate : int
        Audio sample rate in Hz (default 16000).
    output_length : int
        Number of output samples to synthesise (default 32000 = 2 s).
    """

    def __init__(
        self,
        num_delays: int = 16,
        max_delay_ms: float = 50.0,
        sample_rate: int = 16_000,
        output_length: int = DEFAULT_MAX_RIR_LEN,   # 32 000
    ) -> None:
        super().__init__()
        self.num_delays = num_delays
        self.max_delay_ms = max_delay_ms
        self.sample_rate = sample_rate
        self.output_length = output_length

        # Maximum delay in samples
        self.max_delay_samples = int(max_delay_ms * sample_rate / 1000.0)

        # ---- Learnable parameters (unconstrained) ----------------------

        # log_delays: mapped via sigmoid → (0, max_delay_samples)
        #   Initialised so that delays span the log range evenly
        init_delays = torch.linspace(-2.0, 2.0, num_delays)
        self.log_delays = nn.Parameter(init_delays)

        # alpha: per-channel decay rate, sigmoid → (0, 1)
        #   Init near 0.95 (long reverb)
        self.alpha_raw = nn.Parameter(torch.full((num_delays,), 2.94))  # sigmoid(2.94) ≈ 0.95

        # beta: per-channel absorption scale, softplus → (0, ∞)
        #   Controls how fast energy decays per round-trip
        self.beta_raw = nn.Parameter(torch.full((num_delays,), 0.5))

        # Input / output bus gains
        self.input_gains = nn.Parameter(torch.ones(num_delays) / math.sqrt(num_delays))
        self.output_gains = nn.Parameter(torch.ones(num_delays) / math.sqrt(num_delays))

        # ---- Fixed feedback matrix (Hadamard, unitary) -----------------
        self.register_buffer("feedback_matrix", _hadamard_matrix(num_delays))

    # ------------------------------------------------------------------
    #  Constrained parameter accessors
    # ------------------------------------------------------------------

    @property
    def delays_samples(self) -> torch.Tensor:
        """Differentiable delay lengths in samples (float).

        Sigmoid maps log_delays → (0, 1), then scaled to
        (1, max_delay_samples).  Kept as float for differentiability;
        fractional delays are handled via linear interpolation.
        """
        return 1.0 + torch.sigmoid(self.log_delays) * (self.max_delay_samples - 1)

    @property
    def alpha(self) -> torch.Tensor:
        """Per-channel decay rate ∈ (0, 1)."""
        return torch.sigmoid(self.alpha_raw)

    @property
    def beta(self) -> torch.Tensor:
        """Per-channel absorption scale > 0."""
        return F.softplus(self.beta_raw)

    @property
    def decay_gains(self) -> torch.Tensor:
        """Per-channel gain applied each round-trip:
            g_i = alpha_i · exp(-beta_i · m_i / fs)
        """
        m = self.delays_samples                   # [N]
        return self.alpha * torch.exp(-self.beta * m / self.sample_rate)

    # ------------------------------------------------------------------
    #  Fractional delay via linear interpolation
    # ------------------------------------------------------------------

    @staticmethod
    def _frac_delay_read(
        buffer: torch.Tensor,
        write_pos: int,
        delay: torch.Tensor,
        max_len: int,
    ) -> torch.Tensor:
        """Read from a circular buffer at a fractional delay position.

        Uses linear interpolation between the two nearest integer taps
        to keep the operation differentiable.

        Parameters
        ----------
        buffer : Tensor[B, N, max_buf]
            Circular delay-line buffer.
        write_pos : int
            Current write pointer (samples).
        delay : Tensor[N]
            Delay lengths in samples (float, may be fractional).
        max_len : int
            Buffer length.

        Returns
        -------
        out : Tensor[B, N]
        """
        # Integer and fractional parts
        delay_int = delay.long()                          # floor
        frac = delay - delay_int.float()                  # fractional part

        idx0 = (write_pos - delay_int) % max_len          # [N]
        idx1 = (write_pos - delay_int - 1) % max_len      # [N]

        # Gather from buffer:  buffer[:, i, idx[i]]
        B = buffer.size(0)
        idx0_exp = idx0.unsqueeze(0).expand(B, -1)        # [B, N]
        idx1_exp = idx1.unsqueeze(0).expand(B, -1)

        val0 = buffer.gather(2, idx0_exp.unsqueeze(2)).squeeze(2)  # [B, N]
        val1 = buffer.gather(2, idx1_exp.unsqueeze(2)).squeeze(2)

        return (1.0 - frac.unsqueeze(0)) * val0 + frac.unsqueeze(0) * val1

    # ------------------------------------------------------------------
    #  Forward — synthesise the late reverb tail
    # ------------------------------------------------------------------

    def forward(
        self,
        impulse: Optional[torch.Tensor] = None,
        batch_size: int = 1,
    ) -> torch.Tensor:
        """Synthesise a late-reverberation impulse response.

        Parameters
        ----------
        impulse : Tensor[B, output_length], optional
            Excitation signal fed into the FDN.  If *None*, a unit
            impulse (delta at sample 0) is used.
        batch_size : int
            Used only when ``impulse`` is None.

        Returns
        -------
        y : Tensor[B, output_length]
            Synthesised late-reverberation tail.
        """
        device = self.log_delays.device
        N = self.num_delays
        T = self.output_length
        buf_len = self.max_delay_samples + 2  # extra sample for interp

        if impulse is not None:
            B = impulse.size(0)
        else:
            B = batch_size
            impulse = torch.zeros(B, T, device=device)
            impulse[:, 0] = 1.0                           # unit impulse

        # Circular delay-line buffer  [B, N, buf_len]
        # NOTE: we avoid in-place writes (buf[:,:,idx] = ...) because
        # they break autograd after many iterations.  Instead we use
        # scatter to produce a new tensor each step.
        buf = torch.zeros(B, N, buf_len, device=device)
        output_list: list = []

        # Compute constrained parameters inline (avoids property lookup issues)
        delays = 1.0 + torch.sigmoid(self.log_delays) * (self.max_delay_samples - 1)
        alpha = torch.sigmoid(self.alpha_raw)
        beta = F.softplus(self.beta_raw)
        g = alpha * torch.exp(-beta * delays / self.sample_rate)
        A = self.feedback_matrix                          # [N, N]
        b_in = self.input_gains                           # [N]
        c_out = self.output_gains                         # [N]

        # Pre-build scatter index [B, N, 1] — updated each step
        for t in range(T):
            # 1. Read from delay lines (fractional)
            dl_out = self._frac_delay_read(buf, t, delays, buf_len)  # [B, N]

            # 2. Apply per-channel decay
            dl_out = dl_out * g.unsqueeze(0)              # [B, N]

            # 3. Mix through feedback matrix
            fb = torch.matmul(dl_out, A.T)                # [B, N]

            # 4. Inject input
            x_in = impulse[:, t].unsqueeze(1) * b_in.unsqueeze(0)  # [B, N]
            dl_in = fb + x_in

            # 5. Write into delay lines (out-of-place via scatter)
            w_idx = t % buf_len
            idx = torch.full((B, N, 1), w_idx, dtype=torch.long, device=device)
            buf = buf.scatter(2, idx, dl_in.unsqueeze(2))

            # 6. Tap output
            output_list.append((dl_out * c_out.unsqueeze(0)).sum(dim=1))

        return torch.stack(output_list, dim=1)            # [B, T]

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_delay_info(self) -> dict:
        """Return a snapshot of the constrained delay parameters."""
        with torch.no_grad():
            ms = self.delays_samples * 1000.0 / self.sample_rate
            return {
                "delays_ms": ms.cpu().tolist(),
                "delays_samples": self.delays_samples.cpu().tolist(),
                "alpha": self.alpha.cpu().tolist(),
                "beta": self.beta.cpu().tolist(),
                "decay_gains": self.decay_gains.cpu().tolist(),
            }


print("DifferentiableFDN class defined.")
print(f"  Hadamard 16x16 norm check: {_hadamard_matrix(16).norm():.4f} (expect 1.0)")

In [ ]:
# ============================================================
# Cell 17 — Unit Test: Differentiable FDN
# ============================================================

print("=" * 64)
print("  Unit Test — DifferentiableFDN  (Phase 4: FDN)")
print("=" * 64)

# ---- 1. Instantiate ----------------------------------------------------
fdn = DifferentiableFDN(
    num_delays=16,
    max_delay_ms=50.0,
    sample_rate=16_000,
    output_length=DEFAULT_MAX_RIR_LEN,    # 32 000
).to(DEVICE)

print(f"\n  Learnable parameters : {fdn.count_parameters()}")
print(f"  Device               : {DEVICE}")
print(f"  Max delay (samples)  : {fdn.max_delay_samples}")

# ---- 2. Delay constraint verification ----------------------------------
info = fdn.get_delay_info()
delays_ms = info["delays_ms"]
print(f"\n[Delay constraints]")
print(f"  delays (ms)  : {[f'{d:.2f}' for d in delays_ms]}")
print(f"  range        : [{min(delays_ms):.2f}, {max(delays_ms):.2f}] ms")
print(f"  all in (0, 50): {all(0 < d < 50 for d in delays_ms)}")
assert all(0 < d < 50 for d in delays_ms), "Delays must be in (0, 50) ms"

print(f"  alpha range  : [{min(info['alpha']):.4f}, {max(info['alpha']):.4f}]  (expect (0,1))")
assert all(0 < a < 1 for a in info["alpha"]), "Alpha must be in (0, 1)"

print(f"  beta range   : [{min(info['beta']):.4f}, {max(info['beta']):.4f}]  (expect > 0)")
assert all(b > 0 for b in info["beta"]), "Beta must be > 0"

print(f"  decay gains  : [{min(info['decay_gains']):.4f}, {max(info['decay_gains']):.4f}]")

# ---- 3. Forward pass — unit impulse -----------------------------------
BATCH_FDN = 2

with torch.no_grad():
    y = fdn(batch_size=BATCH_FDN)

print(f"\n[Forward — unit impulse]")
print(f"  Output shape : {y.shape}  (expect [{BATCH_FDN}, {DEFAULT_MAX_RIR_LEN}])")
print(f"  Output range : [{y.min().item():.6f}, {y.max().item():.6f}]")
print(f"  Energy (RMS) : {y.pow(2).mean().sqrt().item():.6f}")

assert y.shape == (BATCH_FDN, DEFAULT_MAX_RIR_LEN), f"Shape mismatch: {y.shape}"
assert not torch.isnan(y).any(), "NaN detected in output"
assert not torch.isinf(y).any(), "Inf detected in output"

# ---- 4. Forward pass — custom excitation signal -------------------------
excitation = torch.randn(BATCH_FDN, DEFAULT_MAX_RIR_LEN, device=DEVICE) * 0.01
with torch.no_grad():
    y_exc = fdn(impulse=excitation)

print(f"\n[Forward — noise excitation]")
print(f"  Output shape : {y_exc.shape}")
print(f"  Output range : [{y_exc.min().item():.6f}, {y_exc.max().item():.6f}]")
assert y_exc.shape == (BATCH_FDN, DEFAULT_MAX_RIR_LEN)

# ---- 5. Gradient flow — can we backprop through the FDN? ---------------
print(f"\n[Gradient flow check]")
fdn.train()

# Shorter output for faster gradient test
fdn_short = DifferentiableFDN(
    num_delays=16, max_delay_ms=50.0,
    sample_rate=16_000, output_length=1000,   # 1000 samples for speed
).to(DEVICE)

y_grad = fdn_short(batch_size=1)
loss = y_grad.pow(2).sum()
loss.backward()

grads_ok = True
for name, p in fdn_short.named_parameters():
    if p.grad is None:
        print(f"  WARNING: {name} has no gradient!")
        grads_ok = False
    else:
        g_norm = p.grad.norm().item()
        print(f"  {name:20s}  grad norm = {g_norm:.6f}")

assert grads_ok, "Some parameters did not receive gradients!"

# ---- 6. Feedback matrix is unitary -------------------------------------
A = fdn.feedback_matrix
ident_err = (A @ A.T - torch.eye(16, device=DEVICE)).abs().max().item()
print(f"\n[Feedback matrix unitarity]")
print(f"  ||A A^T - I||_inf   : {ident_err:.2e}  (expect ~0)")
assert ident_err < 1e-5, f"Feedback matrix not unitary: error = {ident_err}"

# ---- 7. Integration with Phase 2 LSTM (if loaded) ---------------------
try:
    model.eval()
    fdn.eval()
    dummy_x = torch.randn(1, INPUT_DIM, device=DEVICE)
    with torch.no_grad():
        edc_pred = model(dummy_x)               # [1, 256, 6]
        rir_late = fdn(batch_size=1)             # [1, 32000]
    print(f"\n[Integration: LSTM → FDN]")
    print(f"  EDC pred shape  : {edc_pred.shape}")
    print(f"  Late reverb shape: {rir_late.shape}")
except NameError:
    print("\n  (Skipping integration — run Phase 2 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — Differentiable FDN is operational.")
print("=" * 64)

---
# Phase 5 — Training Loop

**Goal:** Wire together all preceding phases into an end-to-end training pipeline.

### Data Flow (Training)

```
RIRMegaDataset                   MultibandEDCPredictor
  X [B, 24]  ──────────────────►  forward(X) ──► edc_pred [B, 256, 6]
  Y["edc_mb"] [B, 256, 6]  ──┐                        │
                              └──► PhysicsInformedRIRLoss ──► L_total
                                     │
                              L_edc (RMSE)  +  λ_cont·L_cont  +  λ_mom·L_mom
```

### Training Strategy

| Stage | Epochs | What | Loss |
|:------|:------:|:-----|:-----|
| **Warm-up** | 1 – 20 | LSTM EDC predictor only | EDC RMSE (physics terms off: `lambda=0`) |
| **Physics** | 21 – 50 | LSTM + optional physics | EDC RMSE + continuity + momentum (when collocation points are supplied) |
| **FDN** | Optional | Differentiable FDN fine-tuning | RIR waveform or EDC match against target late tail |

### Hyperparameters

| Param | Default | Notes |
|-------|---------|-------|
| `epochs` | 50 | Adjust for Colab GPU budget |
| `batch_size` | 8 | Increase to 16/32 on large-VRAM GPUs |
| `lr` | 1e-3 | Adam learning rate |
| `weight_decay` | 1e-5 | L2 regularisation |
| `grad_clip` | 1.0 | Max gradient norm |
| `log_every` | 100 | Print loss every N steps |
| `scheduler` | ReduceLROnPlateau | patience=5, factor=0.5 |

In [ ]:
# ============================================================
# Cell 19 — Training Configuration & Trainer
# ============================================================
import time as _time
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    """All hyper-parameters in one place."""

    # ---- Data ----------------------------------------------------------
    batch_size: int = 8
    num_workers: int = 2
    max_rir_len: int = DEFAULT_MAX_RIR_LEN        # 32 000
    sample_rate: int = 16_000

    # ---- LSTM Model ----------------------------------------------------
    hidden_dim: int = 512        # increased for better capacity
    num_layers: int = 3          # deeper LSTM
    num_time_steps: int = 256
    num_bands: int = len(OCTAVE_BANDS)             # 6
    model_dropout: float = 0.05  # reduced dropout for better convergence

    # ---- FDN (optional) ------------------------------------------------
    train_fdn: bool = True
    fdn_num_delays: int = 16
    fdn_max_delay_ms: float = 50.0
    fdn_output_length: int = 3_200                 # 0.5s (sufficient for slope)
    fdn_weight: float = 0.1                        # weight for FDN loss

    # ---- Loss ----------------------------------------------------------
    lambda_cont: float = 0.0                       # off during warm-up
    lambda_mom: float = 0.0

    # ---- Optimiser -----------------------------------------------------
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0

    # ---- Scheduler -----------------------------------------------------
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5

    # ---- Training ------------------------------------------------------
    epochs: int = 50
    log_every: int = 100                           # steps between prints
    val_every_epoch: int = 1

    # ---- AMP (Automatic Mixed Precision) --------------------------------
    use_amp: bool = True                           # enable AMP on GPU

    # ---- Multi-Resolution STFT loss ------------------------------------
    use_mr_stft: bool = False                      # optional; enable explicitly
    mr_stft_weight: float = 1.0
    mr_stft_windows: str = "512,1024,2048"

    # ---- Collocation PINN physics loss ---------------------------------
    use_collocation: bool = False                  # disabled by default; curriculum enables
    collocation_n_points: int = 128
    collocation_lambda_cont: float = 0.01
    collocation_lambda_mom: float = 0.01
    siren_hidden_dim: int = 64
    siren_num_layers: int = 3

    # ---- U-Net refiner -------------------------------------------------
    use_unet: bool = False                         # optional; enable explicitly
    unet_weight: float = 1.0

    # ---- FDN curriculum length (shorter windows speed up early training)
    fdn_curriculum_length: int = 0                 # 0 = disabled
    fdn_curriculum_end_epoch: int = 10


# ======================================================================
#  RIR Trainer
# ======================================================================

class RIRTrainer:
    """End-to-end training harness for the Physics-Informed RIR pipeline.

    Brings together:
      - Phase 1  RIRMegaDataset / DataLoader
      - Phase 2  MultibandEDCPredictor (LSTM)
      - Phase 3  PhysicsInformedRIRLoss
      - Phase 4  DifferentiableFDN (optional)
    """

    def __init__(self, cfg: TrainingConfig) -> None:
        self.cfg = cfg
        self.device = DEVICE

        # ---- Build components ------------------------------------------
        self._build_dataloaders()
        self._build_models()
        self._build_criterion()
        self._build_optimiser()

        # ---- Bookkeeping -----------------------------------------------
        self.global_step = 0
        self.best_val_loss = float("inf")
        self.history: Dict[str, list] = {
            "train_loss": [], "train_edc": [],
            "val_loss": [], "val_edc": [],
            # Extended metrics for curriculum training
            "train_continuity": [], "train_momentum": [],
            "val_continuity": [], "val_momentum": [],
            "train_fdn": [],
        }

    # ------------------------------------------------------------------
    #  Construction helpers
    # ------------------------------------------------------------------

    def _build_dataloaders(self) -> None:
        c = self.cfg
        self.train_loader = get_dataloader(
            split="train", batch_size=c.batch_size,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            num_workers=c.num_workers, shuffle=True,
        )
        self.val_loader = get_dataloader(
            split="val", batch_size=c.batch_size,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            num_workers=c.num_workers, shuffle=False,
        )
        print(f"[Trainer] Train batches: {len(self.train_loader)}, "
              f"Val batches: {len(self.val_loader)}")

    def _build_models(self) -> None:
        c = self.cfg

        self.lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=c.hidden_dim,
            num_layers=c.num_layers,
            num_bands=c.num_bands,
            num_time_steps=c.num_time_steps,
            dropout=c.model_dropout,
        ).to(self.device)
        print(f"[Trainer] LSTM params: {self.lstm.count_parameters():,}")

        if c.train_fdn:
            self.fdn = DifferentiableFDN(
                num_delays=c.fdn_num_delays,
                max_delay_ms=c.fdn_max_delay_ms,
                sample_rate=c.sample_rate,
                output_length=c.fdn_output_length,
            ).to(self.device)
            print(f"[Trainer] FDN  params: {self.fdn.count_parameters():,}")
        else:
            self.fdn = None

        # Optional early-reflection split
        if c.train_fdn and hasattr(c, 'early_late_split') and getattr(c, 'early_late_split', False):
            self.early_refl = EarlyReflectionNet().to(self.device)
        else:
            self.early_refl = None

        # U-Net refiner (optional neural post-processor)
        if c.use_unet:
            self.unet_refiner = UNetRefiner(channels=1).to(self.device)
            print(f"[Trainer] UNet params: {sum(p.numel() for p in self.unet_refiner.parameters()):,}")
        else:
            self.unet_refiner = None

    def _build_criterion(self) -> None:
        self.criterion = PhysicsInformedRIRLoss(
            lambda_cont=self.cfg.lambda_cont,
            lambda_mom=self.cfg.lambda_mom,
        ).to(self.device)

        # MR-STFT loss
        self.mr_stft_loss = None
        if self.cfg.use_mr_stft:
            windows = [int(w) for w in self.cfg.mr_stft_windows.split(",")]
            self.mr_stft_loss = MultiResolutionSTFTLoss(window_lengths=windows).to(self.device)

        # Collocation-based PINN loss with SIREN coordinate network
        self.collocation_loss = None
        if self.cfg.use_collocation:
            coord_net = SIRENCoordinateNet(
                hidden_dim=self.cfg.siren_hidden_dim,
                num_layers=self.cfg.siren_num_layers,
            ).to(self.device)
            self.coord_net = coord_net
            self.collocation_loss = CollocationPhysicsLoss(
                coord_net=coord_net,
                lambda_cont=self.cfg.collocation_lambda_cont,
                lambda_mom=self.cfg.collocation_lambda_mom,
            ).to(self.device)
        else:
            self.coord_net = None

    def _build_optimiser(self) -> None:
        c = self.cfg
        params = list(self.lstm.parameters())
        if self.fdn is not None:
            params += list(self.fdn.parameters())
        if self.early_refl is not None:
            params += list(self.early_refl.parameters())
        if self.unet_refiner is not None:
            params += list(self.unet_refiner.parameters())
        if self.coord_net is not None:
            params += list(self.coord_net.parameters())

        self.optimiser = torch.optim.Adam(
            params, lr=c.lr, weight_decay=c.weight_decay,
        )
        # AMP GradScaler — only active on CUDA
        self.scaler = torch.amp.GradScaler(
            self.device.type, enabled=c.use_amp and self.device.type == "cuda"
        )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimiser,
            mode="min",
            factor=c.scheduler_factor,
            patience=c.scheduler_patience,
        )

    # ------------------------------------------------------------------
    #  Training & validation
    # ------------------------------------------------------------------

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        """Train for one full pass over the training set."""
        self.lstm.train()
        if self.fdn is not None:
            self.fdn.train()

        running = {
            "total": 0.0, "edc": 0.0,
            "continuity": 0.0, "momentum": 0.0,
            "fdn": 0.0,
            "n": 0
        }

        for batch_idx, (x_batch, y_batch) in enumerate(self.train_loader):
            x = x_batch.to(self.device)                                # [B, 24]
            edc_target = y_batch["edc_mb"].to(self.device)             # [B, T, F]

            # ---- Forward (with AMP) ------------------------------------
            self.optimiser.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=self.scaler.is_enabled()):
                edc_pred = self.lstm(x)                                # [B, T, F]

                # ---- Loss --------------------------------------------------
                losses = self.criterion(edc_pred, edc_target)

                # Optional FDN loss: compare FDN output EDC with target
                if self.fdn is not None:
                    fdn_rir = self.fdn(batch_size=x.size(0))           # [B, fdn_len]

                    # Optional U-Net refinement of the FDN output
                    if self.unet_refiner is not None:
                        fdn_rir = self.unet_refiner(fdn_rir.unsqueeze(1)).squeeze(1)

                    # 1. Compute EDC of the generated FDN tail (truncated integration)
                    fdn_energy = torch.cumsum(fdn_rir.flip(1).pow(2), dim=1).flip(1)
                    fdn_edc = fdn_energy / (fdn_energy[:, :1] + 1e-30)
                    fdn_edc_db = 10.0 * torch.log10(fdn_edc + 1e-30)

                    # 2. Slice target RIR to match FDN length & compute matched EDC
                    target_rir_full = y_batch["rir"].to(self.device)   # [B, max_rir_len]
                    target_len = fdn_rir.shape[1]
                    target_slice = target_rir_full[:, :target_len]

                    target_energy = torch.cumsum(target_slice.flip(1).pow(2), dim=1).flip(1)
                    target_edc = target_energy / (target_energy[:, :1] + 1e-30)
                    target_edc_db = 10.0 * torch.log10(target_edc + 1e-30)

                    # 3. Loss (RMSE between dB curves)
                    fdn_loss = torch.sqrt(
                        torch.mean((fdn_edc_db - target_edc_db) ** 2) + 1e-8
                    )
                    losses["total"] = losses["total"] + self.cfg.fdn_weight * fdn_loss
                    losses["fdn"] = fdn_loss

                    # MR-STFT loss on FDN time-domain output
                    if self.mr_stft_loss is not None:
                        mr_loss = self.mr_stft_loss(
                            fdn_rir[:, :target_len].float(),
                            target_slice.float(),
                        )
                        losses["total"] = losses["total"] + self.cfg.mr_stft_weight * mr_loss
                        losses["mr_stft"] = mr_loss

                # Collocation-based PINN physics loss
                if self.collocation_loss is not None:
                    room_dims = x[:, :3].clamp(min=0.1)
                    coll_loss = self.collocation_loss(
                        room_dims, n_points=self.cfg.collocation_n_points
                    )
                    losses["total"] = losses["total"] + coll_loss
                    losses["collocation"] = coll_loss

            # ---- Backward (AMP-aware) ------------------------------------
            self.scaler.scale(losses["total"]).backward()

            # Gradient clipping
            if self.cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(
                    self.lstm.parameters(), self.cfg.grad_clip
                )
                if self.fdn is not None:
                    nn.utils.clip_grad_norm_(
                        self.fdn.parameters(), self.cfg.grad_clip
                    )

            self.scaler.step(self.optimiser)
            self.scaler.update()

            # ---- Logging -----------------------------------------------
            bs = x.size(0)
            running["total"] += losses["total"].item() * bs
            running["edc"] += losses["edc"].item() * bs
            if "continuity" in losses:
                running["continuity"] += losses["continuity"].item() * bs
            if "momentum" in losses:
                running["momentum"] += losses["momentum"].item() * bs
            if "fdn" in losses:
                running["fdn"] += losses["fdn"].item() * bs
            running["n"] += bs
            self.global_step += 1

            if self.global_step % self.cfg.log_every == 0:
                avg_t = running["total"] / running["n"]
                avg_e = running["edc"] / running["n"]
                lr_now = self.optimiser.param_groups[0]["lr"]
                msg = (f"  [Epoch {epoch+1} | Step {self.global_step:>6d}]  "
                       f"loss={avg_t:.4f}  edc={avg_e:.4f}  lr={lr_now:.2e}")
                if self.fdn is not None and "fdn" in losses:
                    msg += f"  fdn={losses['fdn'].item():.4f}"
                print(msg)

        n = running["n"]
        return {
            "total": running["total"] / max(n, 1),
            "edc": running["edc"] / max(n, 1),
            "continuity": running["continuity"] / max(n, 1),
            "momentum": running["momentum"] / max(n, 1),
            "fdn": running["fdn"] / max(n, 1),
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        """Run one full pass over the validation set."""
        self.lstm.eval()
        if self.fdn is not None:
            self.fdn.eval()

        running = {
            "total": 0.0, "edc": 0.0,
            "continuity": 0.0, "momentum": 0.0,
            "n": 0
        }

        for x_batch, y_batch in self.val_loader:
            x = x_batch.to(self.device)
            edc_target = y_batch["edc_mb"].to(self.device)

            edc_pred = self.lstm(x)
            losses = self.criterion(edc_pred, edc_target)

            bs = x.size(0)
            running["total"] += losses["total"].item() * bs
            running["edc"] += losses["edc"].item() * bs
            if "continuity" in losses:
                running["continuity"] += losses["continuity"].item() * bs
            if "momentum" in losses:
                running["momentum"] += losses["momentum"].item() * bs
            running["n"] += bs

        n = running["n"]
        return {
            "total": running["total"] / max(n, 1),
            "edc": running["edc"] / max(n, 1),
            "continuity": running["continuity"] / max(n, 1),
            "momentum": running["momentum"] / max(n, 1),
        }

    # ------------------------------------------------------------------
    def fit(self) -> Dict[str, list]:
        """Full training loop over all epochs."""
        print("=" * 64)
        print("  Training — Physics-Informed RIR Pipeline")
        print("=" * 64)
        print(f"  Device     : {self.device}")
        print(f"  Epochs     : {self.cfg.epochs}")
        print(f"  Batch size : {self.cfg.batch_size}")
        print(f"  LR         : {self.cfg.lr}")
        print(f"  FDN        : {'ON' if self.fdn is not None else 'OFF'}")
        print(f"  Physics λ  : cont={self.cfg.lambda_cont}, "
              f"mom={self.cfg.lambda_mom}")
        print("=" * 64)

        for epoch in range(self.cfg.epochs):
            t0 = _time.time()

            # ---- Train -------------------------------------------------
            train_metrics = self.train_one_epoch(epoch)
            self.history["train_loss"].append(train_metrics["total"])
            self.history["train_edc"].append(train_metrics["edc"])

            # ---- Validate ----------------------------------------------
            if (epoch + 1) % self.cfg.val_every_epoch == 0:
                val_metrics = self.validate()
                self.history["val_loss"].append(val_metrics["total"])
                self.history["val_edc"].append(val_metrics["edc"])

                # LR scheduler step (ReduceLROnPlateau needs metric)
                self.scheduler.step(val_metrics["total"])

                # Best model checkpoint
                if val_metrics["total"] < self.best_val_loss:
                    self.best_val_loss = val_metrics["total"]
                    torch.save(self.lstm.state_dict(), "best_lstm.pt")
                    if self.fdn is not None:
                        torch.save(self.fdn.state_dict(), "best_fdn.pt")
                    if self.unet_refiner is not None:
                        torch.save(self.unet_refiner.state_dict(), "best_unet.pt")
                    marker = " ★ saved"
                else:
                    marker = ""
            else:
                val_metrics = {"total": float("nan"), "edc": float("nan")}
                marker = ""

            elapsed = _time.time() - t0
            print(
                f"Epoch {epoch+1:>3d}/{self.cfg.epochs}  "
                f"train_loss={train_metrics['total']:.4f}  "
                f"val_loss={val_metrics['total']:.4f}  "
                f"({elapsed:.1f}s){marker}"
            )

        print(f"\n{'─' * 64}")
        print(f"  Training complete.  Best val loss: {self.best_val_loss:.4f}")
        print("=" * 64)
        return self.history


print("TrainingConfig and RIRTrainer defined.")

In [ ]:
# ============================================================
# Cell 20 — Launch Training
# ============================================================
# Toggle DRY_RUN = True for a quick smoke-test with synthetic data
# (no HuggingFace download needed).  Set False for real training.
DRY_RUN = True

if DRY_RUN:
    # ------------------------------------------------------------------
    #  Dry-run: validate full pipeline with synthetic tensors
    # ------------------------------------------------------------------
    print("=" * 64)
    print("  DRY RUN — Smoke Test (synthetic data, 2 steps)")
    print("=" * 64)

    B, T, N_BANDS = 4, 256, len(OCTAVE_BANDS)

    # Synthetic batch
    syn_x = torch.randn(B, INPUT_DIM, device=DEVICE)
    syn_edc_mb = torch.randn(B, T, N_BANDS, device=DEVICE) * -30.0   # dB-scale
    syn_edc_broad = torch.randn(B, DEFAULT_MAX_RIR_LEN, device=DEVICE)

    # LSTM
    lstm_dry = MultibandEDCPredictor(
        input_dim=INPUT_DIM, hidden_dim=128, num_layers=1,
        num_bands=N_BANDS, num_time_steps=T, dropout=0.0,
    ).to(DEVICE)

    # Loss
    criterion_dry = PhysicsInformedRIRLoss(
        lambda_cont=0.0, lambda_mom=0.0,
    ).to(DEVICE)

    # Optimiser
    opt_dry = torch.optim.Adam(lstm_dry.parameters(), lr=1e-3)

    # Two training steps
    for step in range(2):
        lstm_dry.train()
        pred = lstm_dry(syn_x)                          # [B, T, N_BANDS]
        losses = criterion_dry(pred, syn_edc_mb)
        opt_dry.zero_grad()
        losses["total"].backward()
        nn.utils.clip_grad_norm_(lstm_dry.parameters(), 1.0)
        opt_dry.step()
        print(f"  Step {step+1}:  loss={losses['total'].item():.4f}  "
              f"edc={losses['edc'].item():.4f}")

    # FDN forward + backward (short output)
    fdn_dry = DifferentiableFDN(
        num_delays=16, max_delay_ms=50.0,
        sample_rate=16_000, output_length=500,
    ).to(DEVICE)
    fdn_out = fdn_dry(batch_size=1)
    fdn_loss = fdn_out.pow(2).mean()
    fdn_loss.backward()
    print(f"  FDN step :  fdn_loss={fdn_loss.item():.6f}")

    # Shape summary
    print(f"\n  LSTM input  : {syn_x.shape}")
    print(f"  LSTM output : {pred.shape}")
    print(f"  FDN output  : {fdn_out.shape}")
    print(f"  Loss keys   : {sorted(losses.keys())}")

    print(f"\n{'─' * 64}")
    print("  ✓  Dry run passed — full pipeline is end-to-end functional.")
    print("=" * 64)
    print("\n  ➜  Set DRY_RUN = False and re-run this cell to train on")
    print("     the real RIRmega dataset.")

else:
    # ------------------------------------------------------------------
    #  Real training on RIRmega dataset
    # ------------------------------------------------------------------
    cfg = TrainingConfig(
        # Data
        batch_size=8,
        num_workers=2,
        # Model
        hidden_dim=256,
        num_layers=2,
        num_time_steps=256,
        model_dropout=0.1,
        # Optional U-Net refiner (set True to activate; see Cell 28)
        use_unet=True,
        # FDN (disabled by default — enable for Phase 4 fine-tuning)
        train_fdn=False,
        # Loss (physics terms off for warm-up; increase after ~20 epochs)
        lambda_cont=0.0,
        lambda_mom=0.0,
        # Optimiser
        lr=1e-3,
        weight_decay=1e-5,
        grad_clip=1.0,
        # Schedule
        epochs=50,
        log_every=100,
    )
    trainer = RIRTrainer(cfg)
    history = trainer.fit()

---
# Phase 6 — LSTM-FDN Bridge, Phase Reconstruction & Full RIR Synthesis

**Goal:** Wire the LSTM's predicted multiband EDC into the FDN's decay parameters, reconstruct time-domain phase, and produce a complete synthesised RIR.

### End-to-End Synthesis Pipeline

```
Room params X [B,24]
       |
  MultibandEDCPredictor (LSTM)
       |
  edc_pred [B, 256, 6]
       |
  ┌────▼────────────────┐
  │  EDC-to-FDN Bridge   │  Maps band decay rates → FDN alpha/beta
  │  (ConditionedFDN)    │  Also predicts delay-line lengths from room size
  └────┬────────────────┘
       |
  FDN late reverb [B, output_length]
       |
  ┌────▼─────────────────────┐
  │  Phase Reconstruction     │  Sticky random sign +
  │  (SignStickyPhase)        │  reverse-differentiation
  └────┬─────────────────────┘
       |
  Full RIR waveform [B, output_length]
```

### Phase Reconstruction: Random Sign-Sticky Reverse Differentiation

EDC gives us the **energy envelope** but not the phase. We reconstruct a plausible time-domain signal:

1. **Energy → Amplitude:** $a(t) = \sqrt{|EDC(t) - EDC(t+1)|}$ (reverse-differentiation of the decay curve)
2. **Random sign:** Multiply each sample by a random $\pm 1$ that "sticks" for short runs (Poisson-distributed flip intervals), mimicking the quasi-random phase of late reverberation
3. **Result:** Perceptually indistinguishable from real late-reverb, while maintaining the exact energy envelope

In [ ]:
# ============================================================
# Cell 22 — LSTM→FDN Conditioning Bridge & Phase Reconstruction
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import Optional, Dict, Tuple


# ======================================================================
#  1.  EDC → FDN parameter mapper
# ======================================================================

class EDCToFDNMapper(nn.Module):
    """Maps a predicted multiband EDC to FDN decay parameters.

    Takes the LSTM's output [B, T, num_bands] and derives per-delay-line
    alpha (decay rate) and beta (absorption scale) values that condition
    the DifferentiableFDN.

    Also takes room dimensions [L, W, H] from the input features to
    predict physically plausible delay-line lengths.
    """

    def __init__(
        self,
        num_bands: int = len(OCTAVE_BANDS),
        num_time_steps: int = 256,
        num_delays: int = 16,
        room_dim: int = 3,                   # L, W, H
    ) -> None:
        super().__init__()
        self.num_delays = num_delays

        # EDC summary: pool over time → [B, num_bands]
        # Then map to FDN parameters
        edc_feat_dim = num_bands * 2         # mean + slope features

        self.edc_encoder = nn.Sequential(
            nn.Linear(edc_feat_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True),
        )

        # Room dims → delay lengths (unconstrained, will be sigmoided)
        self.delay_head = nn.Sequential(
            nn.Linear(room_dim + 64, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_delays),       # → log_delays for FDN
        )

        # EDC features → alpha (per-channel decay rate)
        self.alpha_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_delays),       # pre-sigmoid
        )

        # EDC features → beta (per-channel absorption scale)
        self.beta_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_delays),       # pre-softplus
        )

    def forward(
        self,
        edc_pred: torch.Tensor,              # [B, T, num_bands]
        room_dims: torch.Tensor,             # [B, 3]
    ) -> Dict[str, torch.Tensor]:
        """
        Returns
        -------
        params : dict with keys:
            "log_delays" : [B, num_delays]  (unconstrained)
            "alpha_raw"  : [B, num_delays]  (pre-sigmoid)
            "beta_raw"   : [B, num_delays]  (pre-softplus)
        """
        B = edc_pred.size(0)

        # Extract EDC summary features
        edc_mean = edc_pred.mean(dim=1)                 # [B, bands]
        # Slope: difference between first and last quarter means
        T = edc_pred.size(1)
        q1 = edc_pred[:, :T//4, :].mean(dim=1)         # [B, bands]
        q4 = edc_pred[:, 3*T//4:, :].mean(dim=1)       # [B, bands]
        edc_slope = q4 - q1                              # [B, bands]

        edc_feat = torch.cat([edc_mean, edc_slope], dim=1)  # [B, 2*bands]
        h = self.edc_encoder(edc_feat)                       # [B, 64]

        # Delay lengths conditioned on room geometry
        delay_in = torch.cat([room_dims, h], dim=1)     # [B, 3+64]
        log_delays = self.delay_head(delay_in)           # [B, num_delays]

        # Decay parameters from EDC shape
        alpha_raw = self.alpha_head(h)                   # [B, num_delays]
        beta_raw = self.beta_head(h)                     # [B, num_delays]

        return {
            "log_delays": log_delays,
            "alpha_raw": alpha_raw,
            "beta_raw": beta_raw,
        }


# ======================================================================
#  2.  Conditioned FDN  (FDN whose parameters come from the bridge)
# ======================================================================

class ConditionedFDN(nn.Module):
    """FDN that receives per-sample parameters from the EDC→FDN bridge.

    Unlike the standalone DifferentiableFDN (Phase 4) whose parameters
    are nn.Parameters shared across the batch, this version accepts
    **per-sample** alpha/beta/delay from the conditioning bridge, making
    the reverb tail sample-specific.
    """

    def __init__(
        self,
        num_delays: int = 16,
        max_delay_ms: float = 50.0,
        sample_rate: int = 16_000,
        output_length: int = DEFAULT_MAX_RIR_LEN,
    ) -> None:
        super().__init__()
        self.num_delays = num_delays
        self.max_delay_samples = int(max_delay_ms * sample_rate / 1000.0)
        self.sample_rate = sample_rate
        self.output_length = output_length

        # Fixed feedback matrix and learnable I/O gains
        self.register_buffer("feedback_matrix", _hadamard_matrix(num_delays))
        self.input_gains = nn.Parameter(
            torch.ones(num_delays) / math.sqrt(num_delays)
        )
        self.output_gains = nn.Parameter(
            torch.ones(num_delays) / math.sqrt(num_delays)
        )

    def forward(
        self,
        log_delays: torch.Tensor,       # [B, N]  unconstrained
        alpha_raw: torch.Tensor,         # [B, N]  pre-sigmoid
        beta_raw: torch.Tensor,          # [B, N]  pre-softplus
        impulse: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Returns
        -------
        y : Tensor[B, output_length]
        """
        device = log_delays.device
        B = log_delays.size(0)
        N = self.num_delays
        T = self.output_length
        buf_len = self.max_delay_samples + 2

        # Constrain parameters (per-sample)
        delays = 1.0 + torch.sigmoid(log_delays) * (self.max_delay_samples - 1)  # [B, N]
        alpha = torch.sigmoid(alpha_raw)             # [B, N]
        beta = F.softplus(beta_raw)                  # [B, N]
        g = alpha * torch.exp(-beta * delays / self.sample_rate)  # [B, N]

        A = self.feedback_matrix                     # [N, N]
        b_in = self.input_gains                      # [N]
        c_out = self.output_gains                    # [N]

        if impulse is None:
            impulse = torch.zeros(B, T, device=device)
            impulse[:, 0] = 1.0

        buf = torch.zeros(B, N, buf_len, device=device)
        output_list: list[torch.Tensor] = []

        for t in range(T):
            # Read with per-sample fractional delays
            dl_out = self._batched_frac_read(buf, t, delays, buf_len)  # [B, N]
            dl_out = dl_out * g                          # [B, N]
            fb = torch.matmul(dl_out, A.T)               # [B, N]
            x_in = impulse[:, t].unsqueeze(1) * b_in.unsqueeze(0)
            dl_in = fb + x_in
            w_idx = t % buf_len
            idx = torch.full((B, N, 1), w_idx, dtype=torch.long, device=device)
            buf = buf.scatter(2, idx, dl_in.unsqueeze(2))
            output_list.append((dl_out * c_out.unsqueeze(0)).sum(dim=1))

        return torch.stack(output_list, dim=1)

    @staticmethod
    def _batched_frac_read(
        buf: torch.Tensor,       # [B, N, buf_len]
        write_pos: int,
        delays: torch.Tensor,    # [B, N]  per-sample delays
        buf_len: int,
    ) -> torch.Tensor:
        delay_int = delays.long()
        frac = delays - delay_int.float()
        idx0 = (write_pos - delay_int) % buf_len         # [B, N]
        idx1 = (write_pos - delay_int - 1) % buf_len
        val0 = buf.gather(2, idx0.unsqueeze(2)).squeeze(2)
        val1 = buf.gather(2, idx1.unsqueeze(2)).squeeze(2)
        return (1.0 - frac) * val0 + frac * val1


# ======================================================================
#  3.  Phase Reconstruction — Sticky Random Sign
# ======================================================================

class SignStickyPhaseReconstructor(nn.Module):
    """Reconstruct a time-domain waveform from an energy envelope.

    Algorithm (per sample):
      1. Reverse-differentiate the EDC to get instantaneous energy:
         e(t) = |EDC(t) - EDC(t+1)|
      2. Convert to amplitude: a(t) = sqrt(e(t))
      3. Generate random ±1 signs that "stick" for short runs
         (Poisson-distributed flip intervals with mean `mean_run`),
         mimicking the quasi-random phase of late reverberation.
      4. Multiply: rir(t) = a(t) * sign(t)

    Parameters
    ----------
    mean_run : float
        Mean number of consecutive samples before a sign flip.
        Higher = smoother, lower = more noise-like.
    """

    def __init__(self, mean_run: float = 8.0) -> None:
        super().__init__()
        self.mean_run = mean_run

    @torch.no_grad()
    def _sticky_signs(self, length: int, batch: int, device: torch.device) -> torch.Tensor:
        """Generate sticky ±1 sign sequences [B, L]."""
        signs = torch.ones(batch, length, device=device)
        flip_prob = 1.0 / self.mean_run
        flips = torch.rand(batch, length, device=device) < flip_prob
        # Cumulative XOR via cumsum mod 2
        flip_cumsum = flips.float().cumsum(dim=1)
        signs = 1.0 - 2.0 * (flip_cumsum % 2)
        return signs

    def forward(
        self,
        edc_db: torch.Tensor,         # [B, L]  broadband EDC in dB
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        edc_db : Tensor[B, L]
            Energy Decay Curve in dB (0 dB at t=0, decreasing).

        Returns
        -------
        rir : Tensor[B, L]
            Reconstructed time-domain waveform.
        """
        B, L = edc_db.shape

        # 1. dB → linear energy
        edc_lin = 10.0 ** (edc_db / 10.0)               # [B, L]

        # 2. Reverse-differentiate: instantaneous energy per sample
        #    e(t) = EDC(t) - EDC(t+1), last sample → 0
        e_inst = torch.zeros_like(edc_lin)
        e_inst[:, :-1] = torch.clamp(edc_lin[:, :-1] - edc_lin[:, 1:], min=0.0)

        # 3. Amplitude
        amplitude = torch.sqrt(e_inst + 1e-30)          # [B, L]

        # 4. Sticky random signs
        signs = self._sticky_signs(L, B, edc_db.device)

        return amplitude * signs


# ======================================================================
#  3b.  Multiband Phase Reconstructor  (per-band sign-sticky)
# ======================================================================

class MultibandSignStickyPhaseReconstructor(nn.Module):
    """Applies the Sign-Sticky algorithm per frequency band, then sums bands.

    Applying phase reconstruction on a per-band basis preserves the
    individual decay rates of each octave band, fixing the metallic /
    spiky artefacts that result from a single broadband envelope.

    Parameters
    ----------
    mean_run : float
        Mean number of consecutive samples before a sign flip.
    """

    def __init__(self, mean_run: float = 8.0) -> None:
        super().__init__()
        self.band_recon = SignStickyPhaseReconstructor(mean_run=mean_run)

    def forward(self, edc_mb: torch.Tensor) -> torch.Tensor:
        """Reconstruct per-band waveforms and sum.

        Parameters
        ----------
        edc_mb : Tensor[B, T, num_bands]
            Multiband EDC in dB (same scale as SignStickyPhaseReconstructor).

        Returns
        -------
        rir : Tensor[B, T-1]
            Reconstructed time-domain waveform.
        """
        B, T, num_bands = edc_mb.shape
        band_waveforms = []
        for b in range(num_bands):
            band_edc = edc_mb[:, :, b]              # [B, T]
            band_rir = self.band_recon(band_edc)    # [B, T-1]
            band_waveforms.append(band_rir)
        stacked = torch.stack(band_waveforms, dim=1)  # [B, num_bands, T-1]
        return stacked.sum(dim=1) / num_bands



# ======================================================================
#  4.  Full RIR Synthesiser  (wraps everything)
# ======================================================================

class RIRSynthesiser(nn.Module):
    """End-to-end: room parameters → complete RIR waveform.

    Chains:
      LSTM → EDC-to-FDN bridge → Conditioned FDN → phase reconstruction
    """

    def __init__(
        self,
        lstm: MultibandEDCPredictor,
        num_delays: int = 16,
        max_delay_ms: float = 50.0,
        sample_rate: int = 16_000,
        output_length: int = DEFAULT_MAX_RIR_LEN,
        phase_mean_run: float = 8.0,
        use_unet: bool = False,
        train_fdn: bool = True,
    ) -> None:
        super().__init__()
        self.train_fdn = train_fdn
        self.lstm = lstm
        self.bridge = EDCToFDNMapper(
            num_bands=lstm.num_bands,
            num_time_steps=lstm.num_time_steps,
            num_delays=num_delays,
        )
        self.fdn = ConditionedFDN(
            num_delays=num_delays,
            max_delay_ms=max_delay_ms,
            sample_rate=sample_rate,
            output_length=output_length,
        )
        # Per-band phase reconstruction (replaces single broadband approach)
        self.phase_recon = MultibandSignStickyPhaseReconstructor(mean_run=phase_mean_run)
        # Optional U-Net neural post-processor
        self.unet = UNetRefiner(channels=1) if use_unet else None
        self.output_length = output_length

    def forward(
        self,
        x: torch.Tensor,                    # [B, 24]
        return_intermediates: bool = False,
    ) -> Dict[str, torch.Tensor]:
        """
        Returns
        -------
        result : dict
            "rir"      : [B, output_length]  — final synthesised RIR
            "edc_pred" : [B, T, bands]       — predicted multiband EDC
            "fdn_rir"  : [B, output_length]  — raw FDN output (optional)
        """
        # 1. LSTM predicts multiband EDC  (already in dB-like units:
        #    starts at 0 and decreases via -cumsum(softplus(...)))
        edc_pred = self.lstm(x)                         # [B, T, bands]

        # 2. Bridge maps EDC + room dims → FDN params
        room_dims = x[:, :3]                            # [L, W, H]
        fdn_params = self.bridge(edc_pred, room_dims)

        # 3. Conditioned FDN synthesises late reverb
        fdn_rir = self.fdn(
            log_delays=fdn_params["log_delays"],
            alpha_raw=fdn_params["alpha_raw"],
            beta_raw=fdn_params["beta_raw"],
        )                                                # [B, output_length]

        # 4. Upsample EDC from T (e.g. 256) to output_length so phase
        #    reconstruction produces a full-length waveform.
        #    edc_pred is already in dB — do NOT apply log10 again.
        edc_up = F.interpolate(
            edc_pred.permute(0, 2, 1),                   # [B, bands, T]
            size=self.output_length,
            mode='linear',
            align_corners=True,
        ).permute(0, 2, 1)                               # [B, output_length, bands]
        rir_out = self.phase_recon(edc_up)               # [B, output_length]

        # Combine early (phase reconstruction) and late (FDN) reverb
        if self.train_fdn:
            L = min(rir_out.size(1), fdn_rir.size(1))
            rir = rir_out[:, :L] + fdn_rir[:, :L]
        else:
            rir = rir_out

        # Optional U-Net refinement
        if self.unet is not None:
            rir = self.unet(rir.unsqueeze(1)).squeeze(1)

        # Peak normalization
        rir = rir / (torch.max(torch.abs(rir), dim=-1, keepdim=True).values + 1e-8)

        result = {"rir": rir, "edc_pred": edc_pred}
        if return_intermediates:
            result["fdn_rir"] = fdn_rir
        return result

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("EDCToFDNMapper, ConditionedFDN, SignStickyPhaseReconstructor, "
      "and RIRSynthesiser defined.")

In [ ]:
# ============================================================
# Hotfix — checkpoint compatibility + synthesis correction
# ============================================================
# Run this cell after the Inference cell to patch loading/forward behavior
# without re-editing the earlier large cells.

import pathlib
import re
from typing import Any, Dict


def _infer_lstm_arch_from_ckpt(state: Dict[str, torch.Tensor]) -> Dict[str, int]:
    """Infer LSTM architecture directly from checkpoint tensor shapes."""
    arch = {}
    arch["input_dim"] = int(state["encoder.0.weight"].shape[1])
    arch["hidden_dim"] = int(state["encoder.0.weight"].shape[0])
    arch["num_layers"] = len([k for k in state.keys() if re.fullmatch(r"lstm\.weight_ih_l\d+", k)])
    # output_head.2 is the final Linear in this notebook definition
    arch["num_bands"] = int(state["output_head.2.weight"].shape[0])
    arch["num_time_steps"] = int(state["time_embed"].shape[1])
    return arch


# Forward is now fixed in Cell 22 directly (EDC upsampled, no double-log10).
# No monkey-patching needed.



def load_synthesiser(
    checkpoint_dir: str = ".",
    config: Optional["TrainingConfig"] = None,
    device: str = "cpu",
) -> RIRSynthesiser:
    """Checkpoint-safe loader that auto-matches architecture to saved weights."""
    if config is None:
        config = TrainingConfig()

    ckpt_dir = pathlib.Path(checkpoint_dir)
    lstm_path = ckpt_dir / "best_lstm.pt"

    lstm_state = None
    if lstm_path.exists():
        try:
            lstm_state = torch.load(lstm_path, map_location=device, weights_only=True)
        except Exception as e:
            print(f"⚠ Could not read LSTM checkpoint: {e}")

    if isinstance(lstm_state, dict):
        try:
            inferred = _infer_lstm_arch_from_ckpt(lstm_state)
            print(
                "✓ Inferred LSTM arch from checkpoint: "
                f"hidden={inferred['hidden_dim']}, layers={inferred['num_layers']}, "
                f"T={inferred['num_time_steps']}, bands={inferred['num_bands']}"
            )
            lstm = MultibandEDCPredictor(
                input_dim=inferred["input_dim"],
                hidden_dim=inferred["hidden_dim"],
                num_layers=inferred["num_layers"],
                num_bands=inferred["num_bands"],
                num_time_steps=inferred["num_time_steps"],
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
            lstm.load_state_dict(lstm_state, strict=True)
            print(f"✓ Loaded LSTM weights from {lstm_path}")
        except Exception as e:
            print(f"⚠ Could not load LSTM weights: {e}")
            print("  Falling back to config-defined random LSTM.")
            lstm = MultibandEDCPredictor(
                input_dim=INPUT_DIM,
                hidden_dim=config.hidden_dim,
                num_layers=config.num_layers,
                num_bands=len(OCTAVE_BANDS),
                num_time_steps=config.num_time_steps,
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
    else:
        print(f"⚠ LSTM checkpoint not found at {lstm_path}, using random weights")
        lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=config.hidden_dim,
            num_layers=config.num_layers,
            num_bands=len(OCTAVE_BANDS),
            num_time_steps=config.num_time_steps,
            dropout=getattr(config, "model_dropout", 0.1),
        ).to(device)

    synth = RIRSynthesiser(
        lstm=lstm,
        num_delays=config.fdn_num_delays,
        max_delay_ms=config.fdn_max_delay_ms,
        sample_rate=config.sample_rate,
        output_length=config.max_rir_len,
        use_unet=getattr(config, "use_unet", False),
        train_fdn=getattr(config, "train_fdn", True),
    ).to(device)

    fdn_path = ckpt_dir / "best_fdn.pt"
    if fdn_path.exists():
        try:
            state = torch.load(fdn_path, map_location=device, weights_only=True)
            missing, unexpected = synth.fdn.load_state_dict(state, strict=False)
            print(f"✓ Loaded FDN weights from {fdn_path} (missing={len(missing)}, unexpected={len(unexpected)})")
        except Exception as e:
            print(f"⚠ Could not load FDN weights: {e}")
    else:
        print(f"⚠ FDN checkpoint not found at {fdn_path}, using random weights")

    unet_path = ckpt_dir / "best_unet.pt"
    if getattr(synth, "unet", None) is not None:
        if unet_path.exists():
            try:
                synth.unet.load_state_dict(torch.load(unet_path, map_location=device, weights_only=True), strict=False)
                print(f"✓ Loaded UNet weights from {unet_path}")
            except Exception as e:
                print(f"⚠ Could not load UNet weights: {e}")
        else:
            print(f"ℹ UNet checkpoint not found at {unet_path}; continuing without trained UNet weights")

    return synth



def demo_inference(device: str = str(DEVICE), checkpoint_dir: str = "."):
    """Generate one RIR from made-up room params using best available weights."""
    cfg = TrainingConfig()
    synth = load_synthesiser(checkpoint_dir=checkpoint_dir, config=cfg, device=device)
    synth.eval()

    room = [6.0, 4.0, 3.0]
    result = generate_rir_from_params(
        synth,
        room_dims=room,
        absorption=0.5,
        absorption_bands=[0.3, 0.4, 0.5, 0.55, 0.6, 0.65],
        source_pos=[2.0, 1.5, 1.2],
        mic_pos=[4.0, 2.5, 1.2],
        device=device,
    )

    rir = result["rir"].squeeze(0).numpy()
    has_ckpt = (pathlib.Path(checkpoint_dir) / "best_lstm.pt").exists()
    print(f"Generated RIR shape : {rir.shape}")
    print(f"EDC prediction shape: {result['edc_pred'].shape}")
    print(f"Peak amplitude      : {np.abs(rir).max():.6f}")
    print(f"Model params        : {synth.count_parameters():,}")
    if has_ckpt:
        print("✓ Demo is using checkpoint weights.")
    else:
        print("⚠ Demo is using random weights (train first for meaningful output).")
    return result


print("✓ Hotfix applied: checkpoint-compatible loader + fixed synthesiser forward.")

In [ ]:
# ============================================================
# Cell 23 — Inference & Generation
# ============================================================
"""
Load trained checkpoints and generate RIRs from arbitrary room parameters.
Works stand-alone (no dataset needed) or in batch over the test split.
"""
import json, pathlib, re
from typing import Dict, Optional


@torch.no_grad()
def generate_rir_from_params(
    synthesiser: RIRSynthesiser,
    room_dims: list,          # [L, W, H] in metres
    absorption: float,
    absorption_bands: list,   # 6 octave-band values
    source_pos: list,         # [x, y, z]
    mic_pos: list,            # [x, y, z]
    sample_rate: int = 16_000,
    device: str = "cpu",
) -> dict:
    """Generate a single RIR from physical room parameters.

    Returns a dict with 'rir' [1, L], 'edc_pred' [1, T, bands], etc.
    """
    synthesiser.eval()

    # Compute modal features from room dimensions
    modal_feats = compute_room_modes(*room_dims).tolist()

    # Build the 24-dim input vector (same order as RIRMegaDataset)
    x = (
        room_dims                      # 3
        + source_pos                   # 3
        + mic_pos                      # 3
        + [absorption]                 # 1
        + absorption_bands             # 6
        + modal_feats                  # 8  modal features
    )
    x_t = torch.tensor([x], dtype=torch.float32, device=device)  # [1, 24]
    result = synthesiser(x_t, return_intermediates=True)
    return {k: v.cpu() for k, v in result.items()}


# ------------------------------------------------------------------
# Robust checkpoint loader infering architecture
# ------------------------------------------------------------------

def _infer_lstm_arch_from_ckpt(state: Dict[str, torch.Tensor]) -> Dict[str, int]:
    arch = {}
    arch["input_dim"] = int(state["encoder.0.weight"].shape[1])
    arch["hidden_dim"] = int(state["encoder.0.weight"].shape[0])
    arch["num_layers"] = len([k for k in state.keys() if re.fullmatch(r"lstm\.weight_ih_l\d+", k)])
    arch["num_bands"] = int(state["output_head.2.weight"].shape[0])
    arch["num_time_steps"] = int(state["time_embed"].shape[1])
    return arch


@torch.no_grad()
def load_synthesiser(
    checkpoint_dir: str = ".",
    config: Optional["TrainingConfig"] = None,
    device: str = "cpu",
) -> RIRSynthesiser:
    """Instantiate and load weights for the full synthesis pipeline.

    Automatically adjusts the LSTM size to match saved checkpoints.
    """
    if config is None:
        config = TrainingConfig()

    ckpt_dir = pathlib.Path(checkpoint_dir)
    lstm_path = ckpt_dir / "best_lstm.pt"

    lstm_state = None
    if lstm_path.exists():
        try:
            lstm_state = torch.load(lstm_path, map_location=device, weights_only=True)
        except Exception as e:
            print(f"⚠ Could not read LSTM checkpoint: {e}")

    if isinstance(lstm_state, dict):
        try:
            inf = _infer_lstm_arch_from_ckpt(lstm_state)
            print("✓ inferred LSTM arch from checkpoint", inf)
            lstm = MultibandEDCPredictor(
                input_dim=inf["input_dim"],
                hidden_dim=inf["hidden_dim"],
                num_layers=inf["num_layers"],
                num_bands=inf["num_bands"],
                num_time_steps=inf["num_time_steps"],
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
            lstm.load_state_dict(lstm_state, strict=True)
            print(f"✓ Loaded LSTM weights from {lstm_path}")
        except Exception as e:
            print(f"⚠ Could not load LSTM weights: {e}")
            lstm = MultibandEDCPredictor(
                input_dim=INPUT_DIM,
                hidden_dim=config.hidden_dim,
                num_layers=config.num_layers,
                num_bands=len(OCTAVE_BANDS),
                num_time_steps=config.num_time_steps,
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
    else:
        print(f"⚠ LSTM checkpoint not found at {lstm_path}, using random weights")
        lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=config.hidden_dim,
            num_layers=config.num_layers,
            num_bands=len(OCTAVE_BANDS),
            num_time_steps=config.num_time_steps,
            dropout=getattr(config, "model_dropout", 0.1),
        ).to(device)

    synth = RIRSynthesiser(
        lstm=lstm,
        num_delays=config.fdn_num_delays,
        max_delay_ms=config.fdn_max_delay_ms,
        sample_rate=config.sample_rate,
        output_length=config.max_rir_len,
        use_unet=getattr(config, "use_unet", False),
        train_fdn=getattr(config, "train_fdn", True),
    ).to(device)

    fdn_path = ckpt_dir / "best_fdn.pt"
    if fdn_path.exists():
        try:
            state = torch.load(fdn_path, map_location=device, weights_only=True)
            missing, unexpected = synth.fdn.load_state_dict(state, strict=False)
            print(f"✓ Loaded FDN weights from {fdn_path} (missing={len(missing)}, unexpected={len(unexpected)})")
        except Exception as e:
            print(f"⚠ Could not load FDN weights: {e}")
    else:
        print(f"⚠ FDN checkpoint not found at {fdn_path}, using random weights")

    unet_path = ckpt_dir / "best_unet.pt"
    if getattr(synth, "unet", None) is not None and unet_path.exists():
        try:
            synth.unet.load_state_dict(torch.load(unet_path, map_location=device, weights_only=True), strict=False)
            print(f"✓ Loaded UNet weights from {unet_path}")
        except Exception as e:
            print(f"⚠ Could not load UNet weights: {e}")
    elif getattr(synth, "unet", None) is not None:
        print(f"ℹ UNet checkpoint not found at {unet_path}; continuing without trained UNet weights")

    return synth


# quick inference demo using the robust loader

def demo_inference(device: str = str(DEVICE), checkpoint_dir: str = "."):
    cfg = TrainingConfig()
    synth = load_synthesiser(checkpoint_dir=checkpoint_dir, config=cfg, device=device)
    synth.eval()

    room = [6.0, 4.0, 3.0]
    result = generate_rir_from_params(
        synth,
        room_dims=room,
        absorption=0.5,
        absorption_bands=[0.3, 0.4, 0.5, 0.55, 0.6, 0.65],
        source_pos=[2.0, 1.5, 1.2],
        mic_pos=[4.0, 2.5, 1.2],
        device=device,
    )

    rir = result["rir"].squeeze(0).cpu().numpy()
    print(f"Generated RIR shape : {rir.shape}")
    print(f"EDC prediction shape: {result['edc_pred'].shape}")
    print(f"Peak amplitude      : {np.abs(rir).max():.6f}")
    print(f"Model params        : {synth.count_parameters():,}")
    if (pathlib.Path(checkpoint_dir) / "best_lstm.pt").exists():
        print("✓ Demo is using checkpoint weights.")
    else:
        print("⚠ Demo is using random weights (train first for meaningful output)." )
    return result


print("\n✓ Inference pipeline functional.")

In [ ]:
# ============================================================
# Post-Inference Hotfix (run after Cell 23)
# ============================================================
# Ensures these fixes are active even if Cell 23 was re-run:
# 1) checkpoint-compatible LSTM loader
# 2) corrected synthesiser forward (no double log10)
# 3) RIRTrainer ReduceLROnPlateau safe-step guard

import pathlib
import re


def _infer_lstm_arch_from_ckpt(state):
    return {
        "input_dim": int(state["encoder.0.weight"].shape[1]),
        "hidden_dim": int(state["encoder.0.weight"].shape[0]),
        "num_layers": len([k for k in state.keys() if re.fullmatch(r"lstm\.weight_ih_l\d+", k)]),
        "num_bands": int(state["output_head.2.weight"].shape[0]),
        "num_time_steps": int(state["time_embed"].shape[1]),
    }


# Forward is now fixed in Cell 22 directly (EDC upsampled, no double-log10).
# No monkey-patching needed.


# Guard old RIRTrainer.fit() implementations that call scheduler.step() with no metric.
_old_rirtrainer_fit = RIRTrainer.fit

def _rirtrainer_fit_guarded(self):
    if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        _orig_step = self.scheduler.step

        def _safe_step(metrics=None, epoch=None):
            if metrics is None:
                if isinstance(getattr(self, "history", None), dict) and self.history.get("val_loss"):
                    metrics = self.history["val_loss"][-1]
                else:
                    metrics = float("inf")
            return _orig_step(metrics, epoch)

        self.scheduler.step = _safe_step
    return _old_rirtrainer_fit(self)


RIRTrainer.fit = _rirtrainer_fit_guarded



def load_synthesiser(checkpoint_dir=".", config=None, device="cpu"):
    if config is None:
        config = TrainingConfig()

    ckpt_dir = pathlib.Path(checkpoint_dir)
    lstm_path = ckpt_dir / "best_lstm.pt"

    lstm_state = None
    if lstm_path.exists():
        try:
            lstm_state = torch.load(lstm_path, map_location=device, weights_only=True)
        except Exception as e:
            print(f"⚠ Could not read LSTM checkpoint: {e}")

    if isinstance(lstm_state, dict):
        try:
            a = _infer_lstm_arch_from_ckpt(lstm_state)
            lstm = MultibandEDCPredictor(
                input_dim=a["input_dim"],
                hidden_dim=a["hidden_dim"],
                num_layers=a["num_layers"],
                num_bands=a["num_bands"],
                num_time_steps=a["num_time_steps"],
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
            lstm.load_state_dict(lstm_state, strict=True)
            print(f"✓ Loaded LSTM weights from {lstm_path} with inferred arch")
        except Exception as e:
            print(f"⚠ Could not load LSTM weights: {e}")
            lstm = MultibandEDCPredictor(
                input_dim=INPUT_DIM,
                hidden_dim=config.hidden_dim,
                num_layers=config.num_layers,
                num_bands=len(OCTAVE_BANDS),
                num_time_steps=config.num_time_steps,
                dropout=getattr(config, "model_dropout", 0.1),
            ).to(device)
    else:
        print(f"⚠ LSTM checkpoint not found at {lstm_path}, using random weights")
        lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=config.hidden_dim,
            num_layers=config.num_layers,
            num_bands=len(OCTAVE_BANDS),
            num_time_steps=config.num_time_steps,
            dropout=getattr(config, "model_dropout", 0.1),
        ).to(device)

    synth = RIRSynthesiser(
        lstm=lstm,
        num_delays=config.fdn_num_delays,
        max_delay_ms=config.fdn_max_delay_ms,
        sample_rate=config.sample_rate,
        output_length=config.max_rir_len,
        use_unet=getattr(config, "use_unet", False),
        train_fdn=getattr(config, "train_fdn", True),
    ).to(device)

    fdn_path = ckpt_dir / "best_fdn.pt"
    if fdn_path.exists():
        try:
            s = torch.load(fdn_path, map_location=device, weights_only=True)
            missing, unexpected = synth.fdn.load_state_dict(s, strict=False)
            print(f"✓ Loaded FDN weights from {fdn_path} (missing={len(missing)}, unexpected={len(unexpected)})")
        except Exception as e:
            print(f"⚠ Could not load FDN weights: {e}")

    unet_path = ckpt_dir / "best_unet.pt"
    if getattr(synth, "unet", None) is not None and unet_path.exists():
        try:
            synth.unet.load_state_dict(torch.load(unet_path, map_location=device, weights_only=True), strict=False)
            print(f"✓ Loaded UNet weights from {unet_path}")
        except Exception as e:
            print(f"⚠ Could not load UNet weights: {e}")
    elif getattr(synth, "unet", None) is not None:
        print(f"ℹ UNet checkpoint not found at {unet_path}; continuing without trained UNet weights")

    return synth



def demo_inference(device: str = str(DEVICE), checkpoint_dir: str = "."):
    cfg = TrainingConfig()
    synth = load_synthesiser(checkpoint_dir=checkpoint_dir, config=cfg, device=device)
    synth.eval()

    result = generate_rir_from_params(
        synth,
        room_dims=[6.0, 4.0, 3.0],
        absorption=0.5,
        absorption_bands=[0.3, 0.4, 0.5, 0.55, 0.6, 0.65],
        source_pos=[2.0, 1.5, 1.2],
        mic_pos=[4.0, 2.5, 1.2],
        device=device,
    )

    rir = result["rir"].squeeze(0).cpu().numpy()
    print(f"Generated RIR shape : {rir.shape}")
    print(f"EDC prediction shape: {result['edc_pred'].shape}")
    print(f"Peak amplitude      : {np.abs(rir).max():.6f}")
    print(f"Model params        : {synth.count_parameters():,}")
    if (pathlib.Path(checkpoint_dir) / "best_lstm.pt").exists():
        print("✓ Demo is using checkpoint weights.")
    else:
        print("⚠ Demo is using random weights (train first for meaningful output).")
    return result


print("✓ Post-inference hotfix active.")

In [ ]:
# ============================================================
# Cell 24 — Evaluation Metrics
# ============================================================
"""
Compute quantitative evaluation metrics on the test set:
  • RT60 error (s)
  • Spectral distance (log-spectral, dB)
  • EDC RMSE (dB)
  • DRR error (dB)
"""
from scipy.signal import fftconvolve


# ── RT60 estimation from an RIR waveform ─────────────────────
def estimate_rt60(rir: np.ndarray, sample_rate: int = 16_000) -> float:
    """Estimate RT60 from a 1-D RIR using Schroeder integration.

    Uses T20 extrapolation: find the time to decay from −5 dB to −25 dB
    (a 20-dB window), then extrapolate to 60 dB.
    """
    rir = rir / (np.abs(rir).max() + 1e-30)
    energy = rir ** 2
    edc = np.cumsum(energy[::-1])[::-1]
    edc = edc / (edc[0] + 1e-30)
    edc_db = 10.0 * np.log10(edc + 1e-30)

    # Find −5 dB and −25 dB crossings
    idx_5 = np.where(edc_db <= -5.0)[0]
    idx_25 = np.where(edc_db <= -25.0)[0]
    if len(idx_5) == 0 or len(idx_25) == 0:
        return float("nan")

    t5 = idx_5[0] / sample_rate
    t25 = idx_25[0] / sample_rate
    dt = t25 - t5
    if dt <= 0:
        return float("nan")
    rt60 = 3.0 * dt               # extrapolate 20 dB → 60 dB
    return float(rt60)


# ── Log-spectral distance ────────────────────────────────────
def log_spectral_distance(
    rir_pred: np.ndarray,
    rir_ref: np.ndarray,
    n_fft: int = 2048,
) -> float:
    """RMS of log magnitude spectra difference (dB)."""
    S_pred = np.abs(np.fft.rfft(rir_pred, n=n_fft)) + 1e-30
    S_ref = np.abs(np.fft.rfft(rir_ref, n=n_fft)) + 1e-30
    lsd = np.sqrt(np.mean((20 * np.log10(S_pred) - 20 * np.log10(S_ref)) ** 2))
    return float(lsd)


# ── EDC RMSE ─────────────────────────────────────────────────
def edc_rmse_db(rir_pred: np.ndarray, rir_ref: np.ndarray) -> float:
    """RMSE between EDCs in dB."""
    def _edc(x):
        e = x ** 2
        edc = np.cumsum(e[::-1])[::-1]
        edc = edc / (edc[0] + 1e-30)
        return 10.0 * np.log10(edc + 1e-30)

    length = min(len(rir_pred), len(rir_ref))
    edc_p = _edc(rir_pred[:length])
    edc_r = _edc(rir_ref[:length])
    return float(np.sqrt(np.mean((edc_p - edc_r) ** 2)))


# ── DRR (Direct-to-Reverberant Ratio) ────────────────────────
def compute_drr(rir: np.ndarray, sample_rate: int = 16_000, direct_ms: float = 2.5) -> float:
    """DRR in dB.  Direct = first `direct_ms` ms starting at index 0.

    For synthetic RIRs the direct sound window starts strictly at index 0 to
    avoid out-of-bounds NaNs that result from argmax-based shifting.
    """
    n_direct = int((direct_ms / 1000.0) * sample_rate)
    n_direct = max(1, min(n_direct, len(rir)))
    e_direct = np.sum(rir[:n_direct] ** 2) + 1e-30
    e_reverb = np.sum(rir[n_direct:] ** 2) + 1e-30
    return float(10.0 * np.log10(e_direct / e_reverb))


# ── Full test-set evaluation ─────────────────────────────────
@torch.no_grad()
def evaluate_on_test_set(
    synthesiser: RIRSynthesiser,
    test_loader: torch.utils.data.DataLoader,
    sample_rate: int = 16_000,
    device: str = "cpu",
    max_batches: Optional[int] = None,
) -> dict:
    """Evaluate synthesiser on the test split.

    Returns dict of metric_name → list of per-sample values.
    """
    synthesiser.eval()
    metrics = {
        "rt60_error": [],
        "lsd": [],
        "edc_rmse": [],
        "drr_error": [],
    }

    for i, (x_batch, y_batch) in enumerate(test_loader):
        if max_batches and i >= max_batches:
            break
        x_batch = x_batch.to(device)
        result = synthesiser(x_batch, return_intermediates=True)
        rir_pred = result["rir"].cpu().numpy()
        rir_ref = y_batch["rir"].numpy()

        B = rir_pred.shape[0]
        for b in range(B):
            rp = rir_pred[b]
            rr = rir_ref[b]
            length = int(y_batch["rir_length"][b].item())
            rr = rr[:length]
            min_len = min(len(rp), len(rr))
            rp = rp[:min_len]
            rr = rr[:min_len]

            rt60_p = estimate_rt60(rp, sample_rate)
            rt60_r = estimate_rt60(rr, sample_rate)
            if not (np.isnan(rt60_p) or np.isnan(rt60_r)):
                metrics["rt60_error"].append(abs(rt60_p - rt60_r))

            metrics["lsd"].append(log_spectral_distance(rp, rr))
            metrics["edc_rmse"].append(edc_rmse_db(rp, rr))

            drr_p = compute_drr(rp, sample_rate)
            drr_r = compute_drr(rr, sample_rate)
            metrics["drr_error"].append(abs(drr_p - drr_r))

    # Aggregate
    summary = {}
    for k, v in metrics.items():
        if len(v) > 0:
            arr = np.array(v)
            summary[k] = {
                "mean": float(arr.mean()),
                "std": float(arr.std()),
                "median": float(np.median(arr)),
                "n": len(v),
            }
        else:
            summary[k] = {"mean": float("nan"), "std": 0, "median": float("nan"), "n": 0}

    # Pretty-print
    print("\n" + "=" * 60)
    print("  TEST-SET EVALUATION RESULTS")
    print("=" * 60)
    for k, s in summary.items():
        unit = "s" if "rt60" in k else "dB"
        print(f"  {k:15s}:  mean={s['mean']:.4f} ± {s['std']:.4f} {unit}  "
              f"(median={s['median']:.4f}, n={s['n']})")
    print("=" * 60 + "\n")
    return summary


print("Evaluation functions defined: estimate_rt60, log_spectral_distance, "
      "edc_rmse_db, compute_drr, evaluate_on_test_set")

In [ ]:
# ============================================================
# Cell 25 — Visualization (Conference-Quality Figures)
# ============================================================
"""
Plotting helpers for:
  1. Training loss curves (EDC / physics / FDN / total)
  2. Predicted vs ground-truth multiband EDC
  3. Generated RIR waveform comparison
  4. EDC + RT60 overlay with −5/−25/−60 dB markers
  5. Conference-style results summary table
  6. Spectrogram comparison
  7. Per-band RT60 comparison (predicted vs reference)
"""
import matplotlib
# Use inline backend for Colab; fall back to Agg for headless
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Conference figure style ──────────────────────────────────
CONFERENCE_STYLE = {
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 1.5,
}
plt.rcParams.update(CONFERENCE_STYLE)


# ── 1. Training loss curves ──────────────────────────────────
def plot_training_curves(
    history: dict,
    title: str = "Training History",
    save_path: Optional[str] = None,
) -> None:
    """Plot training and validation loss curves with component breakdown.

    Parameters
    ----------
    history : dict with keys like 'train_loss', 'val_loss', 'train_edc',
              'val_edc', 'train_physics', 'train_fdn', etc.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Panel 1: Total loss ──────────────────────────────────
    ax = axes[0]
    if "train_loss" in history and history["train_loss"]:
        epochs = range(1, len(history["train_loss"]) + 1)
        ax.plot(epochs, history["train_loss"], label="Train Total",
                linewidth=2, color="#2196F3")
    if "val_loss" in history and history["val_loss"]:
        epochs = range(1, len(history["val_loss"]) + 1)
        ax.plot(epochs, history["val_loss"], label="Val Total",
                linewidth=2, linestyle="--", color="#F44336")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Total Loss")
    ax.legend()

    # ── Panel 2: Component breakdown ─────────────────────────
    ax = axes[1]
    component_keys = [k for k in history
                      if k not in ("train_loss", "val_loss")
                      and isinstance(history[k], list)
                      and len(history[k]) > 0]
    colours = ["#4CAF50", "#FF9800", "#9C27B0", "#00BCD4", "#795548", "#E91E63"]
    if component_keys:
        for i, key in enumerate(sorted(component_keys)):
            c = colours[i % len(colours)]
            epochs = range(1, len(history[key]) + 1)
            ax.plot(epochs, history[key], label=key, linewidth=1.5, color=c)
        ax.legend(fontsize=8, ncol=2)
    else:
        ax.text(0.5, 0.5, "No component breakdown\navailable",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=11, color="gray")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss Component")
    ax.set_title("Loss Breakdown")

    # ── Panel 3: Learning rate / convergence indicator ───────
    ax = axes[2]
    if "train_loss" in history and len(history["train_loss"]) > 1:
        losses = history["train_loss"]
        smoothed = []
        window = min(5, len(losses))
        for i in range(len(losses)):
            start = max(0, i - window + 1)
            smoothed.append(sum(losses[start:i+1]) / (i - start + 1))
        epochs = range(1, len(smoothed) + 1)
        ax.plot(epochs, smoothed, label="Smoothed Train Loss",
                linewidth=2, color="#2196F3")
        if "val_loss" in history and history["val_loss"]:
            val_losses = history["val_loss"]
            val_smoothed = []
            for i in range(len(val_losses)):
                start = max(0, i - window + 1)
                val_smoothed.append(
                    sum(val_losses[start:i+1]) / (i - start + 1))
            val_epochs = range(1, len(val_smoothed) + 1)
            ax.plot(val_epochs, val_smoothed, label="Smoothed Val Loss",
                    linewidth=2, linestyle="--", color="#F44336")
        ax.legend()
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Smoothed Loss")
    ax.set_title("Convergence (moving avg)")

    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# ── 2. Multiband EDC comparison ──────────────────────────────
def plot_multiband_edc(
    edc_pred: np.ndarray,       # [T, bands]
    edc_ref: Optional[np.ndarray] = None,   # [T, bands]
    band_names: Optional[list] = None,
    title: str = "Multiband EDC",
    save_path: Optional[str] = None,
) -> None:
    """Plot predicted (solid) vs reference (dashed) multiband EDC."""
    if band_names is None:
        band_names = [f"{b} Hz" for b in OCTAVE_BANDS]

    T, B = edc_pred.shape
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, B))

    for b in range(B):
        ax.plot(edc_pred[:, b], color=colors[b], linewidth=1.8,
                label=f"Pred {band_names[b]}")
        if edc_ref is not None:
            ax.plot(edc_ref[:, b], color=colors[b], linewidth=1.2,
                    linestyle="--", alpha=0.7, label=f"Ref {band_names[b]}")

    ax.set_xlabel("Time Steps")
    ax.set_ylabel("EDC (dB)")
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 3. RIR waveform comparison ───────────────────────────────
def plot_rir_waveform(
    rir_pred: np.ndarray,
    rir_ref: Optional[np.ndarray] = None,
    sample_rate: int = 16_000,
    title: str = "Generated RIR",
    save_path: Optional[str] = None,
) -> None:
    """Plot predicted and (optionally) reference RIR waveforms."""
    n_axes = 2 if rir_ref is not None else 1
    fig, axes = plt.subplots(n_axes, 1,
                              figsize=(14, 3.5 * n_axes),
                              sharex=True)
    if rir_ref is None:
        axes = [axes]

    t_pred = np.arange(len(rir_pred)) / sample_rate
    axes[0].plot(t_pred, rir_pred, color="#2196F3", linewidth=0.4, alpha=0.85)
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(f"{title} — Predicted")

    if rir_ref is not None:
        t_ref = np.arange(len(rir_ref)) / sample_rate
        axes[1].plot(t_ref, rir_ref, color="#F44336", linewidth=0.4, alpha=0.85)
        axes[1].set_ylabel("Amplitude")
        axes[1].set_title(f"{title} — Reference")

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 4. EDC with RT60 markers ─────────────────────────────────
def plot_edc_with_rt60(
    rir: np.ndarray,
    sample_rate: int = 16_000,
    title: str = "EDC & RT60",
    save_path: Optional[str] = None,
) -> None:
    """Plot the broadband EDC with −5/−25/−60 dB markers."""
    energy = rir ** 2
    edc = np.cumsum(energy[::-1])[::-1]
    edc = edc / (edc[0] + 1e-30)
    edc_db = 10.0 * np.log10(edc + 1e-30)
    t = np.arange(len(rir)) / sample_rate

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, edc_db, color="navy", linewidth=1.5)

    for level, colour in [(-5, "#4CAF50"), (-25, "#FF9800"), (-60, "#F44336")]:
        ax.axhline(y=level, color=colour, linestyle=":", alpha=0.6,
                    label=f"{level} dB")
        idx = np.where(edc_db <= level)[0]
        if len(idx) > 0:
            ax.axvline(x=idx[0] / sample_rate, color=colour,
                        linestyle=":", alpha=0.4)

    rt60 = estimate_rt60(rir, sample_rate)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("EDC (dB)")
    ax.set_title(f"{title}  |  RT60 ≈ {rt60:.3f} s")
    ax.set_ylim([-80, 5])
    ax.legend()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 5. Spectrogram comparison ────────────────────────────────
def plot_spectrogram_comparison(
    rir_pred: np.ndarray,
    rir_ref: np.ndarray,
    sample_rate: int = 16_000,
    title: str = "Spectrogram Comparison",
    save_path: Optional[str] = None,
) -> None:
    """Side-by-side spectrogram of predicted vs reference RIR."""
    from scipy.signal import spectrogram as sp_spectrogram

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    nperseg = min(256, len(rir_pred) // 4)

    for ax, rir, label, cmap in [
        (axes[0], rir_pred, "Predicted", "viridis"),
        (axes[1], rir_ref, "Reference", "viridis"),
    ]:
        f, t, Sxx = sp_spectrogram(rir, fs=sample_rate, nperseg=nperseg)
        Sxx_db = 10.0 * np.log10(Sxx + 1e-30)
        im = ax.pcolormesh(t, f, Sxx_db, shading="gouraud", cmap=cmap)
        ax.set_ylabel("Frequency (Hz)")
        ax.set_xlabel("Time (s)")
        ax.set_title(f"{label}")
        fig.colorbar(im, ax=ax, label="Power (dB)")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 6. Conference results summary table ──────────────────────
def plot_results_table(
    metrics: dict,
    title: str = "Evaluation Results Summary",
    save_path: Optional[str] = None,
) -> None:
    """Render a conference-style results table as a figure.

    Parameters
    ----------
    metrics : dict with keys like 'rt60_error_mean', 'lsd_mean',
              'edc_rmse_mean', 'drr_mean', etc.
    """
    # Build rows: metric name, value, unit, target range
    rows = []
    metric_defs = [
        ("RT60 Error",    "rt60_error_mean", "s",  "< 0.10"),
        ("RT60 Std",      "rt60_error_std",  "s",  "< 0.05"),
        ("LSD",           "lsd_mean",        "dB", "< 5.0"),
        ("EDC RMSE",      "edc_rmse_mean",   "dB", "< 5.0"),
        ("DRR",           "drr_mean",        "dB", "—"),
        ("DRR Std",       "drr_std",         "dB", "—"),
    ]
    for label, key, unit, target in metric_defs:
        val = metrics.get(key, metrics.get(key.replace("_mean", ""), None))
        if val is not None:
            rows.append([label, f"{val:.4f}", unit, target])
        else:
            rows.append([label, "N/A", unit, target])

    fig, ax = plt.subplots(figsize=(8, 2 + 0.4 * len(rows)))
    ax.axis("off")
    table = ax.table(
        cellText=rows,
        colLabels=["Metric", "Value", "Unit", "Target"],
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.0, 1.6)

    # Style header
    for j in range(4):
        cell = table[0, j]
        cell.set_facecolor("#2196F3")
        cell.set_text_props(color="white", fontweight="bold")

    # Colour-code values against targets
    for i, (_, _, _, target) in enumerate(metric_defs):
        if target != "—" and rows[i][1] != "N/A":
            try:
                val = float(rows[i][1])
                thresh = float(target.replace("< ", ""))
                colour = "#C8E6C9" if val < thresh else "#FFCDD2"
                table[i + 1, 1].set_facecolor(colour)
            except ValueError:
                pass

    ax.set_title(title, fontsize=14, fontweight="bold", pad=20)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 7. Per-band RT60 bar chart ───────────────────────────────
def plot_per_band_rt60(
    rt60_pred: np.ndarray,
    rt60_ref: np.ndarray,
    band_names: Optional[list] = None,
    title: str = "Per-Band RT60 Comparison",
    save_path: Optional[str] = None,
) -> None:
    """Grouped bar chart comparing predicted vs reference RT60 per band."""
    if band_names is None:
        band_names = [f"{b} Hz" for b in OCTAVE_BANDS]

    n = len(band_names)
    x = np.arange(n)
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - w / 2, rt60_pred[:n], w, label="Predicted",
           color="#2196F3", alpha=0.85)
    ax.bar(x + w / 2, rt60_ref[:n], w, label="Reference",
           color="#F44336", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(band_names, rotation=30)
    ax.set_ylabel("RT60 (s)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


# ── 8. Quick demo: visualise the inference result ────────────
def visualise_demo(demo_result: dict, sample_rate: int = 16_000):
    """Visualise results from demo_inference()."""
    rir = demo_result["rir"].squeeze(0).numpy()
    edc_pred = demo_result["edc_pred"].squeeze(0).numpy()

    plot_rir_waveform(rir, sample_rate=sample_rate,
                      title="Demo Generated RIR")
    plot_edc_with_rt60(rir, sample_rate=sample_rate,
                       title="Demo EDC & RT60")
    plot_multiband_edc(edc_pred, title="Demo Predicted Multiband EDC")


print("✓ All visualisation functions defined (conference-quality).")

try:
    visualise_demo(demo_result)
except Exception as e:
    print(f"Visualisation skipped (run Cell 22 first): {e}")

In [ ]:
# ============================================================
# Cell 26 — Curriculum Training Schedule & Full Pipeline Execution
# ============================================================
"""
Curriculum schedule:
  Phase A  (epochs 1-10)  : EDC RMSE only        → λ_cont = λ_mom = 0
  Phase B  (epochs 11-25) : + physics ramped in   → λ linearly 0 → target
  Phase C  (epochs 26-40) : + FDN co-training     → freeze physics λ, add FDN
  Phase D  (epochs 41-50) : fine-tune everything  → reduce LR, all losses on

The CurriculumTrainer subclasses RIRTrainer and overrides `fit()`.
"""


@dataclass
class CurriculumConfig(TrainingConfig):
    """Extends TrainingConfig with curriculum schedule parameters."""

    # Phase boundaries (start epoch, 0-indexed)
    warmup_epochs: int = 15         # Phase A: EDC only (longer warmup for better base)
    physics_ramp_end: int = 30      # Phase B ends
    fdn_phase_start: int = 30       # Phase C
    finetune_start: int = 50        # Phase D
    
    # Target physics lambdas (reached at end of Phase B)
    # Keep physics terms very small — they are regularisers, not primary objectives
    lambda_cont_target: float = 0.01
    lambda_mom_target: float = 0.005

    # FDN settings for Phase C onward
    train_fdn: bool = True          # will be activated in Phase C
    fdn_weight: float = 0.05        # small weight to not destabilise training

    # Fine-tune LR multiplier
    finetune_lr_factor: float = 0.1

    # Total epochs — 75 gives better convergence without being too slow
    epochs: int = 75


class CurriculumTrainer(RIRTrainer):
    """RIRTrainer with a curriculum schedule for physics-informed training.

    Automatically adjusts loss weights and model configuration based on
    the current epoch, following the 4-phase schedule above.
    """

    def __init__(self, cfg: CurriculumConfig) -> None:
        # Start in Phase A: no physics, no FDN
        cfg.lambda_cont = 0.0
        cfg.lambda_mom = 0.0
        # Don't build FDN yet — will be created in Phase C
        cfg_init = CurriculumConfig(**{
            k: v for k, v in vars(cfg).items()
        })
        cfg_init.train_fdn = False
        super().__init__(cfg_init)
        self.full_cfg = cfg        # keep the original with all phases
        self._fdn_activated = False

    def _update_schedule(self, epoch: int) -> str:
        """Adjust lambdas, FDN, and LR based on the current epoch.

        Returns a string indicating the current phase.
        """
        c = self.full_cfg

        # Phase A: warmup — EDC only (disable collocation too)
        if epoch < c.warmup_epochs:
            self.criterion.lambda_cont = 0.0
            self.criterion.lambda_mom = 0.0
            # Gate collocation loss off during warmup
            if hasattr(self, '_colloc_backup') and self._colloc_backup is not None:
                pass  # already disabled
            elif self.collocation_loss is not None:
                self._colloc_backup = self.collocation_loss
                self.collocation_loss = None
            return f"A (EDC only, λ=0)"

        # Phase B: ramp physics (re-enable collocation if configured)
        elif epoch < c.physics_ramp_end:
            # Re-enable collocation loss if it was disabled during warmup
            if hasattr(self, '_colloc_backup') and self._colloc_backup is not None:
                self.collocation_loss = self._colloc_backup
                self._colloc_backup = None
            progress = (epoch - c.warmup_epochs) / max(
                c.physics_ramp_end - c.warmup_epochs, 1
            )
            self.criterion.lambda_cont = c.lambda_cont_target * progress
            self.criterion.lambda_mom = c.lambda_mom_target * progress
            return (f"B (physics ramp, λ_cont={self.criterion.lambda_cont:.4f}, "
                    f"λ_mom={self.criterion.lambda_mom:.4f})")

        # Phase C: add FDN co-training
        elif epoch < c.finetune_start:
            self.criterion.lambda_cont = c.lambda_cont_target
            self.criterion.lambda_mom = c.lambda_mom_target

            if not self._fdn_activated and c.train_fdn:
                self._activate_fdn()

            return f"C (+ FDN, λ frozen)"

        # Phase D: fine-tune everything
        else:
            self.criterion.lambda_cont = c.lambda_cont_target
            self.criterion.lambda_mom = c.lambda_mom_target

            if not self._fdn_activated and c.train_fdn:
                self._activate_fdn()

            return f"D (fine-tune, all losses)"

    def _activate_fdn(self) -> None:
        """Lazily build and attach the FDN + bridge for co-training."""
        c = self.full_cfg
        self.fdn = DifferentiableFDN(
            num_delays=c.fdn_num_delays,
            max_delay_ms=c.fdn_max_delay_ms,
            sample_rate=c.sample_rate,
            output_length=c.fdn_output_length,
        ).to(self.device)

        # Add FDN params to the optimiser
        self.optimiser.add_param_group({
            "params": list(self.fdn.parameters()),
            "lr": self.optimiser.param_groups[0]["lr"],
        })
        # Keep ReduceLROnPlateau in sync: extend min_lrs for the new param group
        if hasattr(self.scheduler, "min_lrs"):
            self.scheduler.min_lrs.append(self.scheduler.min_lrs[0])
        self.cfg.train_fdn = True
        self.cfg.fdn_weight = c.fdn_weight
        self._fdn_activated = True
        print(f"[Curriculum] FDN activated — "
              f"{self.fdn.count_parameters():,} params added to optimiser")

    def _reduce_lr(self, factor: float) -> None:
        """Reduce LR for all param groups."""
        for pg in self.optimiser.param_groups:
            pg["lr"] *= factor
        print(f"[Curriculum] LR reduced by factor {factor:.2f} → "
              f"{self.optimiser.param_groups[0]['lr']:.2e}")

    def fit(self) -> Dict[str, list]:
        """Full training loop with curriculum schedule."""
        c = self.full_cfg

        print("=" * 64)
        print("  Curriculum Training — Physics-Informed RIR Pipeline")
        print("=" * 64)
        print(f"  Device      : {self.device}")
        print(f"  Total epochs: {c.epochs}")
        print(f"  Phase A     : epochs  1-{c.warmup_epochs}   (EDC only)")
        print(f"  Phase B     : epochs {c.warmup_epochs+1}-{c.physics_ramp_end}  "
              f"(physics ramp)")
        print(f"  Phase C     : epochs {c.physics_ramp_end+1}-{c.finetune_start} "
              f"(+ FDN co-training)")
        print(f"  Phase D     : epochs {c.finetune_start+1}-{c.epochs}  "
              f"(fine-tune)")
        print("=" * 64)

        lr_reduced = False

        for epoch in range(c.epochs):
            t0 = _time.time()

            # ---- Update curriculum -------------------------------------
            phase_str = self._update_schedule(epoch)

            # Reduce LR once at fine-tune phase
            if epoch == c.finetune_start and not lr_reduced:
                self._reduce_lr(c.finetune_lr_factor)
                lr_reduced = True

            # ---- Train -------------------------------------------------
            train_metrics = self.train_one_epoch(epoch)
            self.history["train_loss"].append(train_metrics["total"])
            self.history["train_edc"].append(train_metrics["edc"])
            self.history["train_continuity"].append(train_metrics["continuity"])
            self.history["train_momentum"].append(train_metrics["momentum"])
            if train_metrics["fdn"] > 0:
                self.history["train_fdn"].append(train_metrics["fdn"])

            # ---- Validate ----------------------------------------------
            if (epoch + 1) % self.cfg.val_every_epoch == 0:
                val_metrics = self.validate()
                self.history["val_loss"].append(val_metrics["total"])
                self.history["val_edc"].append(val_metrics["edc"])
                self.history["val_continuity"].append(val_metrics["continuity"])
                self.history["val_momentum"].append(val_metrics["momentum"])

                self.scheduler.step(val_metrics["total"])

                if val_metrics["total"] < self.best_val_loss:
                    self.best_val_loss = val_metrics["total"]
                    torch.save(self.lstm.state_dict(), "best_lstm.pt")
                    if self.fdn is not None:
                        torch.save(self.fdn.state_dict(), "best_fdn.pt")
                    if getattr(self, 'unet_refiner', None) is not None:
                        torch.save(self.unet_refiner.state_dict(), "best_unet.pt")
                    marker = " ★ saved"
                else:
                    marker = ""
            else:
                val_metrics = {"total": float("nan")}
                marker = ""

            elapsed = _time.time() - t0
            print(
                f"Epoch {epoch+1:>3d}/{c.epochs}  [{phase_str}]  "
                f"train={train_metrics['total']:.4f}  "
                f"val={val_metrics['total']:.4f}  "
                f"({elapsed:.1f}s){marker}"
            )

        print(f"\n{'─' * 64}")
        print(f"  Curriculum training complete.  Best val loss: "
              f"{self.best_val_loss:.4f}")
        print("=" * 64)
        return self.history


print("CurriculumConfig and CurriculumTrainer defined.")

In [ ]:
# ============================================================
# Cell 27 — Full Pipeline Execution
# ============================================================
"""
Toggle DRY_RUN to True for a quick smoke test (random data, 2 steps).
Set to False for full curriculum training on the RIRmega dataset.
"""

DRY_RUN = False   # ← Set to False for real training on Colab GPU

if DRY_RUN:
    print("=" * 64)
    print("  DRY RUN — Smoke-testing the full pipeline")
    print("=" * 64)

    # 1. Quick curriculum trainer test
    dry_cfg = CurriculumConfig(
        batch_size=2, epochs=3,
        warmup_epochs=1, physics_ramp_end=2,
        fdn_phase_start=2, finetune_start=3,
        hidden_dim=64, num_layers=1,
        num_time_steps=32, fdn_output_length=500,
        train_fdn=False,            # skip FDN for speed in DRY_RUN
        lambda_cont_target=0.01,
        lambda_mom_target=0.005,
    )

    # Use synthetic data instead of downloading
    device = str(DEVICE)
    print("\n--- Testing RIRSynthesiser (random weights) ---")
    lstm = MultibandEDCPredictor(
        input_dim=INPUT_DIM,
        hidden_dim=dry_cfg.hidden_dim,
        num_layers=dry_cfg.num_layers,
        num_time_steps=dry_cfg.num_time_steps,
    ).to(device)
    synth = RIRSynthesiser(lstm=lstm, output_length=2000).to(device)

    x_syn = torch.randn(2, INPUT_DIM, device=device)
    out = synth(x_syn, return_intermediates=True)
    print(f"  RIR shape      : {out['rir'].shape}")
    print(f"  EDC pred shape : {out['edc_pred'].shape}")
    print(f"  FDN RIR shape  : {out['fdn_rir'].shape}")
    print(f"  Total params   : {synth.count_parameters():,}")

    # 2. Test evaluation metrics
    print("\n--- Testing evaluation metrics ---")
    rir_fake1 = np.random.randn(2000) * np.exp(-np.linspace(0, 5, 2000))
    rir_fake2 = np.random.randn(2000) * np.exp(-np.linspace(0, 4, 2000))
    rt60 = estimate_rt60(rir_fake1, 16000)
    lsd = log_spectral_distance(rir_fake1, rir_fake2)
    edc_err = edc_rmse_db(rir_fake1, rir_fake2)
    drr = compute_drr(rir_fake1)
    print(f"  RT60 estimate  : {rt60:.3f} s")
    print(f"  LSD            : {lsd:.2f} dB")
    print(f"  EDC RMSE       : {edc_err:.2f} dB")
    print(f"  DRR            : {drr:.2f} dB")

    # 3. Test visualisation
    print("\n--- Testing visualisation ---")
    rir_test = out["rir"][0].detach().cpu().numpy()
    edc_test = out["edc_pred"][0].detach().cpu().numpy()
    plot_rir_waveform(rir_test, title="Dry Run — Synthesised RIR")
    plot_edc_with_rt60(rir_test, title="Dry Run — EDC")
    plot_multiband_edc(edc_test, title="Dry Run — Multiband EDC")

    # 4. Test backward pass through full synth
    print("\n--- Testing backward pass ---")
    synth.train()
    x_syn2 = torch.randn(2, INPUT_DIM, device=device, requires_grad=True)
    out2 = synth(x_syn2)
    loss = out2["rir"].pow(2).mean() + out2["edc_pred"].pow(2).mean()
    loss.backward()
    grad_ok = x_syn2.grad is not None and x_syn2.grad.abs().sum() > 0
    print(f"  Loss           : {loss.item():.6f}")
    print(f"  Gradient flows : {grad_ok}")

    print("\n" + "=" * 64)
    print("  DRY RUN COMPLETE — All pipeline components functional ✓")
    print("=" * 64)

else:
    # ══════════════════════════════════════════════════════════
    #  Full curriculum training on RIRmega dataset
    # ══════════════════════════════════════════════════════════
    print("Starting full curriculum training on RIRmega dataset...")
    print("(This may take ~1.5–2.5 hours on a Colab T4 GPU.)\n")

    cfg = CurriculumConfig(
        # Data — batch 16 balances speed vs GPU memory on T4/L4
        batch_size=16,
        epochs=50,
        # Model
        hidden_dim=256,
        num_layers=2,
        num_time_steps=256,
        # Curriculum schedule
        warmup_epochs=12,           # longer EDC-only warmup for stable start
        physics_ramp_end=30,        # gentler physics ramp
        fdn_phase_start=30,
        finetune_start=42,
        # Physics lambdas — reduced to avoid overwhelming EDC loss
        lambda_cont_target=0.01,    # was 0.1 — too aggressive
        lambda_mom_target=0.005,    # was 0.05 — too aggressive
        # FDN
        train_fdn=True,
        fdn_weight=0.05,            # halved — let EDC lead
        fdn_output_length=3_200,    # shorter FDN = faster + less memory
        # Optimiser
        lr=1e-3,
        grad_clip=1.0,
    )

    trainer = CurriculumTrainer(cfg)
    history = trainer.fit()

    # ── Save checkpoints & history to Drive ───────────────────
    print("\n--- Saving to Google Drive ---")
    try:
        save_checkpoint(trainer.lstm.state_dict(), "best_lstm.pt")
        if trainer.fdn is not None:
            save_checkpoint(trainer.fdn.state_dict(), "best_fdn.pt")
        save_history(history)
    except Exception as e:
        print(f"  ⚠ Drive save failed ({e}), files remain in working dir")

    # ══════════════════════════════════════════════════════════
    #  Post-training evaluation on held-out test set
    # ══════════════════════════════════════════════════════════
    print("\n--- Loading best checkpoint and evaluating on test set ---")
    synth = load_synthesiser(checkpoint_dir=".", config=cfg, device=str(DEVICE))

    test_loader = get_dataloader(
        split="test", batch_size=cfg.batch_size,
        max_rir_len=cfg.max_rir_len,
        num_time_steps=cfg.num_time_steps,
        sample_rate=cfg.sample_rate,
        num_workers=cfg.num_workers, shuffle=False,
    )
    test_metrics = evaluate_on_test_set(
        synth, test_loader, sample_rate=cfg.sample_rate, device=str(DEVICE)
    )

    # Save metrics to Drive
    try:
        save_metrics(test_metrics)
    except Exception:
        pass

    # ══════════════════════════════════════════════════════════
    #  Conference-quality figures & results
    # ══════════════════════════════════════════════════════════
    print("\n--- Generating conference-quality figures ---")

    # ── 1. Training curves (3-panel: total, breakdown, convergence) ──
    plot_training_curves(history, title="Curriculum Training History",
                         save_path="training_curves.png")

    # ── 2. Results summary table ─────────────────────────────
    # Flatten nested metrics dict for the table plotter
    flat_metrics = {}
    for k, v in test_metrics.items():
        if isinstance(v, dict):
            for stat, val in v.items():
                flat_metrics[f"{k}_{stat}"] = val
        else:
            flat_metrics[k] = v
    plot_results_table(flat_metrics, title="Test-Set Evaluation Results",
                       save_path="results_table.png")

    # ── 3. Free memory before inference ──────────────────────
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    synth.eval()
    sample_count = 0
    for i, (x_batch, y_batch) in enumerate(test_loader):
        if i >= 2:
            break
        # Use small batch to avoid OOM during FDN synthesis
        x_batch = x_batch[:4].to(str(DEVICE))
        y_batch = {k: v[:4] for k, v in y_batch.items()}
        with torch.no_grad():
            result = synth(x_batch, return_intermediates=True)

        for b in range(min(2, x_batch.size(0))):
            idx = i * 4 + b       # sequential sample index
            rir_p = result["rir"][b].cpu().numpy()
            rir_r = y_batch["rir"][b].numpy()[:int(y_batch["rir_length"][b])]
            edc_p = result["edc_pred"][b].cpu().numpy()

            # Trim to common length for fair comparison
            min_len = min(len(rir_p), len(rir_r))
            rir_p_cmp = rir_p[:min_len]
            rir_r_cmp = rir_r[:min_len]

            # ── 4. RIR waveform comparison ───────────────────
            fname_waveform = f"rir_sample_{idx}.png"
            plot_rir_waveform(rir_p_cmp, rir_ref=rir_r_cmp,
                              title=f"Test Sample {idx}",
                              save_path=fname_waveform)

            # ── 5. EDC overlay: predicted vs reference ───────
            plot_edc_with_rt60(rir_p_cmp,
                               title=f"Predicted EDC — Sample {idx}",
                               save_path=f"edc_pred_{idx}.png")
            plot_edc_with_rt60(rir_r_cmp,
                               title=f"Reference EDC — Sample {idx}",
                               save_path=f"edc_ref_{idx}.png")

            # ── 6. Multiband EDC (predicted) ─────────────────
            plot_multiband_edc(edc_p,
                               title=f"Multiband EDC — Sample {idx}",
                               save_path=f"edc_mb_{idx}.png")

            # ── 7. Spectrogram comparison ────────────────────
            try:
                plot_spectrogram_comparison(
                    rir_p_cmp, rir_r_cmp,
                    sample_rate=cfg.sample_rate,
                    title=f"Spectrogram — Sample {idx}",
                    save_path=f"spectrogram_{idx}.png",
                )
            except Exception as e:
                print(f"  Spectrogram skipped for sample {idx}: {e}")

            # ── 8. Per-sample metrics annotation ─────────────
            rt60_p = estimate_rt60(rir_p_cmp, cfg.sample_rate)
            rt60_r = estimate_rt60(rir_r_cmp, cfg.sample_rate)
            lsd_val = log_spectral_distance(rir_p_cmp, rir_r_cmp)
            edc_val = edc_rmse_db(rir_p_cmp, rir_r_cmp)
            drr_p = compute_drr(rir_p_cmp, cfg.sample_rate)
            drr_r = compute_drr(rir_r_cmp, cfg.sample_rate)
            print(f"  Sample {idx}: RT60 pred={rt60_p:.3f}s ref={rt60_r:.3f}s "
                  f"| LSD={lsd_val:.2f}dB | EDC RMSE={edc_val:.2f}dB "
                  f"| DRR pred={drr_p:.1f}dB ref={drr_r:.1f}dB")

            # ── 9. Save all outputs to Drive ─────────────────
            try:
                for fn in [fname_waveform, f"edc_pred_{idx}.png",
                           f"edc_ref_{idx}.png", f"edc_mb_{idx}.png",
                           f"spectrogram_{idx}.png"]:
                    if os.path.exists(fn):
                        save_figure(fn, fn)
                save_rir_audio(rir_p, cfg.sample_rate,
                               f"generated_rir_{idx}.wav")
                save_rir_audio(rir_r, cfg.sample_rate,
                               f"reference_rir_{idx}.wav")
            except Exception:
                pass
            sample_count += 1

    print(f"\n  Generated {sample_count} sample comparisons.")

    # ── 10. Save summary figures to Drive ─────────────────────
    try:
        for fn in ["training_curves.png", "results_table.png"]:
            if os.path.exists(fn):
                save_figure(fn, fn)
    except Exception:
        pass

    # ── Backup notebook ───────────────────────────────────────
    try:
        print(f"\n✓ All outputs saved to: {SAVE_DIR}")
        backup_notebook()
    except Exception:
        pass

    print("\n" + "=" * 64)
    print("  ✓ Full pipeline execution complete.")
    print("    Figures saved at 300 DPI — ready for conference paper.")
    print("=" * 64)

In [ ]:
# ============================================================
# Cell 28 — UNet Refiner (Optional Post-processing)
# ============================================================
"""
Optional U-Net-based refinement module that can post-process the
synthesised RIR to improve fine spectral detail.

This is an *optional* module — the core pipeline works without it.
Set USE_UNET_REFINER = True in the Full Pipeline cell to activate.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class ConvBlock1D(nn.Module):
    """Conv1D → GroupNorm → ReLU (× 2).

    GroupNorm is used instead of BatchNorm so the block works correctly
    even when batch_size = 1 (BatchNorm requires B > 1 in training mode).
    """

    def __init__(self, in_ch: int, out_ch: int, kernel: int = 3) -> None:
        super().__init__()
        pad = kernel // 2
        # Groups=min(out_ch, 8) keeps norm well-conditioned for typical channel counts
        num_groups = min(out_ch, 8) if out_ch >= 8 else 1
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=pad),
            nn.GroupNorm(num_groups, out_ch),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel, padding=pad),
            nn.GroupNorm(num_groups, out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.conv = ConvBlock1D(in_ch, out_ch)
        self.pool = nn.MaxPool1d(2)

    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose1d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock1D(out_ch * 2, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.size(-1) != skip.size(-1):
            x = F.interpolate(x, size=skip.size(-1))
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SinusoidalPosEncoding(nn.Module):
    """Sinusoidal positional encoding for 1D sequences."""

    def __init__(self, dim: int, max_len: int = 32_000) -> None:
        super().__init__()
        pe = torch.zeros(1, dim, max_len)
        pos = torch.arange(max_len).unsqueeze(0).float()
        div = torch.exp(
            torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim)
        )
        pe[0, 0::2, :] = torch.sin(pos * div.unsqueeze(1))
        pe[0, 1::2, :] = torch.cos(pos * div.unsqueeze(1))
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :, : x.size(-1)]


class MultiHeadAttentionBottleneck(nn.Module):
    """Self-attention bottleneck for the U-Net bridge."""

    def __init__(self, dim: int, heads: int = 4) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : [B, C, L]  →  attn expects [B, L, C]
        xT = x.permute(0, 2, 1)
        out, _ = self.attn(xT, xT, xT)
        out = self.norm(out + xT)
        return out.permute(0, 2, 1)


class UNetRefiner(nn.Module):
    """Compact U-Net for RIR post-processing.

    Takes a raw generated RIR [B, 1, L] and outputs a refined version.
    Optional: use when you want to sharpen early-reflection detail.
    """

    def __init__(self, channels: int = 1) -> None:
        super().__init__()
        ch = [channels, 16, 32, 64, 128]

        self.encoders = nn.ModuleList([
            EncoderBlock(ch[i], ch[i + 1])
            for i in range(len(ch) - 1)
        ])
        self.pos_enc = SinusoidalPosEncoding(ch[-1])
        self.bottleneck = nn.Sequential(
            ConvBlock1D(ch[-1], ch[-1]),
            MultiHeadAttentionBottleneck(ch[-1]),
        )
        self.decoders = nn.ModuleList([
            DecoderBlock(ch[i + 1], ch[i])
            for i in range(len(ch) - 2, 0, -1)
        ])
        self.head = nn.Conv1d(ch[1], channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)

        x = self.pos_enc(x)
        x = self.bottleneck(x)

        for dec, skip in zip(self.decoders, reversed(skips[:-1])):
            x = dec(x, skip)

        return self.head(x)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── Quick sanity check ──────────────────────────────────────
print("UNetRefiner class defined.")
refiner = UNetRefiner()
print(f"  Parameters: {refiner.count_parameters():,}")
x_test = torch.randn(2, 1, 4096)
with torch.no_grad():
    y_test = refiner(x_test)
print(f"  Input  shape: {x_test.shape}")
print(f"  Output shape: {y_test.shape}")
assert y_test.shape == x_test.shape, f"Shape mismatch: {y_test.shape} != {x_test.shape}"
print("  ✓ UNet Refiner operational.")


In [ ]:
# ============================================================
# Cell 29 — Interactive Live Demo (ipywidgets)
# ============================================================
"""
Interactive demo: adjust room parameters with sliders, generate an RIR,
and compare it against a Sabine analytical reference.

Usage:
  1. Run all cells above first.
  2. Run this cell — sliders will appear.
  3. Move a slider or click Generate → synthesised RIR updates.

Requirements: ipywidgets (usually pre-installed in Colab).
"""
import os
import numpy as np
import torch
import torch.nn.functional as F
import IPython.display as ipd

try:
    import ipywidgets as widgets
    from ipywidgets import VBox, HBox, Label, Layout
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print("ipywidgets not available — run: !pip install ipywidgets")


# ── Analytical reference RIR (Sabine stochastic model) ─────────────

def _sabine_reference_rir(length, width, height, absorption,
                          src_pos, mic_pos, sr=16_000, rir_len=32_000):
    """Generate an analytical Sabine-model reference RIR.

    The envelope decays exponentially with time constant derived from
    Sabine RT60.  A direct-sound pulse is placed at the propagation
    delay, and a handful of first-order image-source reflections are
    included for early-reflection structure.

    Parameters
    ----------
    Returns
    -------
    rir : ndarray[rir_len]  peak-normalised reference RIR
    rt60 : float  Sabine RT60 in seconds
    """
    V = length * width * height
    S = 2 * (length * width + width * height + height * length)
    rt60 = 0.161 * V / (absorption * S + 1e-6)
    tau = rt60 / (3 * np.log(10))  # exponential decay constant

    t = np.arange(rir_len, dtype=np.float32) / sr
    envelope = np.exp(-t / max(tau, 1e-6))

    # Stochastic late reverb: Gaussian noise shaped by envelope
    rng = np.random.default_rng(42)  # fixed seed for reproducibility
    rir = (rng.standard_normal(rir_len).astype(np.float32) * envelope)

    # Direct sound
    dist = np.sqrt(sum((s - m) ** 2 for s, m in zip(src_pos, mic_pos)))
    n_direct = max(1, int(dist / 343.0 * sr))
    if n_direct < rir_len:
        rir[n_direct] += 1.0 / (4 * np.pi * dist + 1e-6)

    # First-order image sources (6 walls)
    for dim in range(3):
        for sign in (-1, +1):
            img = list(src_pos)
            room = [length, width, height]
            if sign > 0:
                img[dim] = 2 * room[dim] - src_pos[dim]
            else:
                img[dim] = -src_pos[dim]
            d = np.sqrt(sum((img[k] - mic_pos[k]) ** 2 for k in range(3)))
            n = int(d / 343.0 * sr)
            if 0 <= n < rir_len:
                refl_coeff = np.sqrt(1 - absorption)
                rir[n] += refl_coeff / (4 * np.pi * d + 1e-6)

    # Peak normalise
    rir = rir / (np.abs(rir).max() + 1e-8)
    return rir, rt60


# ── Acoustic metrics ─────────────────────────────────────────────────

def _edc_from_rir(rir, sr):
    """Compute Schroeder EDC (dB) and estimate RT60."""
    h2 = rir.astype(np.float64) ** 2
    edc = np.cumsum(h2[::-1])[::-1].copy()
    edc_db = 10 * np.log10(edc / (edc[0] + 1e-30) + 1e-30)
    try:
        t5 = np.where(edc_db <= -5)[0][0] / sr
        t25 = np.where(edc_db <= -25)[0][0] / sr
        rt60_est = 3.0 * (t25 - t5)
    except IndexError:
        rt60_est = float("nan")
    return edc_db, rt60_est


def _compute_drr(rir, sr=16_000, direct_ms=2.5):
    n = max(1, min(int(direct_ms / 1000 * sr), len(rir)))
    e_d = np.sum(rir[:n] ** 2) + 1e-30
    e_r = np.sum(rir[n:] ** 2) + 1e-30
    return 10 * np.log10(e_d / e_r)


def _compute_clarity(rir, sr=16_000, t_ms=50):
    n = max(1, min(int(t_ms / 1000 * sr), len(rir)))
    e_early = np.sum(rir[:n] ** 2) + 1e-30
    e_late = np.sum(rir[n:] ** 2) + 1e-30
    return 10 * np.log10(e_early / e_late)


# ── Build / load synthesiser using checkpoint-compatible loader ──────

def _build_demo_synthesiser(device=None):
    """Load the synthesiser with architecture inferred from checkpoint."""
    if device is None:
        device = DEVICE
    save_dir = SAVE_DIR if "SAVE_DIR" in dir() else "."
    try:
        synth = load_synthesiser(
            checkpoint_dir=save_dir, config=None, device=str(device)
        )
    except Exception as e:
        print(f"⚠ load_synthesiser failed ({e}), building with defaults")
        lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM, hidden_dim=256, num_layers=2,
        ).to(device)
        synth = RIRSynthesiser(
            lstm=lstm, output_length=DEFAULT_MAX_RIR_LEN,
            use_unet=False, train_fdn=True,
        ).to(device)
    synth.eval()
    return synth


def _build_input_vector(length, width, height, absorption, src_x, src_y, src_h,
                        mic_x, mic_y, mic_h):
    """Construct the 24-dim input feature vector from slider values."""
    room_vol = length * width * height
    room_surf = 2 * (length * width + width * height + height * length)
    sabine_rt60 = 0.161 * room_vol / (absorption * room_surf + 1e-6)
    mean_free_path = 4.0 * room_vol / room_surf
    aspect_lw = length / (width + 1e-6)
    aspect_lh = length / (height + 1e-6)
    aspect_wh = width / (height + 1e-6)

    # 8 modal features
    modal_feats = compute_room_modes(length, width, height).tolist()

    base_feats = [
        length, width, height, room_vol, room_surf,
        sabine_rt60, mean_free_path, aspect_lw, aspect_lh, aspect_wh,
        src_x, src_y, src_h, mic_x, mic_y, mic_h,
    ][:16]
    feats = base_feats + modal_feats   # total 24
    return torch.tensor(feats, dtype=torch.float32, device=DEVICE).unsqueeze(0)


# ── Main demo function ─────────────────────────────────────────────────

def run_live_demo():
    """Launch the interactive RIR generation demo with reference comparison."""
    if not _HAS_WIDGETS:
        print("Install ipywidgets first: !pip install ipywidgets")
        return

    synth = _build_demo_synthesiser(DEVICE)
    SR = 16_000

    style  = {"description_width": "130px"}
    layout = Layout(width="500px")

    # Room geometry sliders
    sl_L   = widgets.FloatSlider(value=5.0, min=2.0, max=20.0, step=0.5,
                                 description="Length (m)", style=style, layout=layout)
    sl_W   = widgets.FloatSlider(value=4.0, min=2.0, max=15.0, step=0.5,
                                 description="Width (m)",  style=style, layout=layout)
    sl_H   = widgets.FloatSlider(value=3.0, min=2.0, max=6.0,  step=0.25,
                                 description="Height (m)", style=style, layout=layout)
    sl_abs = widgets.FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05,
                                 description="Absorption", style=style, layout=layout,
                                 readout_format=".2f")

    # Source position
    sl_sx  = widgets.FloatSlider(value=2.0, min=0.5, max=18.0, step=0.5,
                                 description="Src X (m)", style=style, layout=layout)
    sl_sy  = widgets.FloatSlider(value=2.0, min=0.5, max=14.0, step=0.5,
                                 description="Src Y (m)", style=style, layout=layout)
    sl_sh  = widgets.FloatSlider(value=1.5, min=0.5, max=3.0,  step=0.25,
                                 description="Src H (m)", style=style, layout=layout)

    out_audio   = widgets.Output()
    out_metrics = widgets.Output()

    def on_generate(_):
        L, W, H = sl_L.value, sl_W.value, sl_H.value
        # Clamp source position inside room
        sx = min(sl_sx.value, L - 0.5)
        sy = min(sl_sy.value, W - 0.5)
        sh = min(sl_sh.value, H - 0.3)
        # Fixed mic at room centre
        mx, my, mh = L / 2, W / 2, 1.5

        x_vec = _build_input_vector(L, W, H, sl_abs.value, sx, sy, sh, mx, my, mh)

        with torch.no_grad():
            result = synth(x_vec, return_intermediates=True)

        rir_pred = result["rir"].squeeze().cpu().numpy().astype(np.float32)

        # Generate Sabine analytical reference
        src_pos = [sx, sy, sh]
        mic_pos = [mx, my, mh]
        rir_ref, sabine_rt60 = _sabine_reference_rir(
            L, W, H, sl_abs.value, src_pos, mic_pos, sr=SR, rir_len=len(rir_pred),
        )

        # ── Compute metrics for both ─────────────────────────
        edc_pred_db, rt60_pred = _edc_from_rir(rir_pred, SR)
        edc_ref_db, rt60_ref = _edc_from_rir(rir_ref, SR)
        drr_pred = _compute_drr(rir_pred, SR)
        drr_ref = _compute_drr(rir_ref, SR)
        c50_pred = _compute_clarity(rir_pred, SR, 50)
        c50_ref = _compute_clarity(rir_ref, SR, 50)
        c80_pred = _compute_clarity(rir_pred, SR, 80)
        c80_ref = _compute_clarity(rir_ref, SR, 80)
        drr_diff = abs(drr_pred - drr_ref)

        # ── Update outputs ────────────────────────────────────
        out_metrics.clear_output(wait=True)
        out_audio.clear_output(wait=True)

        with out_metrics:
            from IPython.display import HTML, display
            vol = L * W * H
            html = f"""<div style='font-family:monospace;font-size:13px;padding:10px;
                                   border:1px solid #555;border-radius:6px;
                                   background:transparent;color:inherit;
                                   max-width:600px'>
              <b style='font-size:14px'>Room Parameters</b><br>
              Room: {L:.1f} &times; {W:.1f} &times; {H:.1f} m&sup3; = {vol:.1f} m&sup3;<br>
              Absorption coeff: {sl_abs.value:.2f}<br>
              Source pos: ({sx:.1f}, {sy:.1f}, {sh:.1f}) m
              &nbsp;&nbsp;Mic pos: ({mx:.1f}, {my:.1f}, {mh:.1f}) m<br>
              <hr style='border-color:#555;margin:6px 0'>
              <b style='font-size:14px'>Acoustic Metrics Comparison</b>
              <table style='width:100%;border-collapse:collapse;margin-top:4px;
                            color:inherit'>
                <tr style='border-bottom:1px solid #555'>
                  <th style='text-align:left;padding:3px'>Metric</th>
                  <th style='text-align:right;padding:3px'>Predicted</th>
                  <th style='text-align:right;padding:3px'>Reference</th>
                  <th style='text-align:right;padding:3px'>|&Delta;|</th>
                </tr>
                <tr><td style='padding:3px'>RT60 (s)</td>
                    <td style='text-align:right;padding:3px'>{rt60_pred:.3f}</td>
                    <td style='text-align:right;padding:3px'>{rt60_ref:.3f}</td>
                    <td style='text-align:right;padding:3px'>{abs(rt60_pred-rt60_ref):.3f}</td></tr>
                <tr><td style='padding:3px'>Sabine RT60 (s)</td>
                    <td style='text-align:right;padding:3px' colspan='3'>{sabine_rt60:.3f}</td></tr>
                <tr><td style='padding:3px'>DRR (dB)</td>
                    <td style='text-align:right;padding:3px'>{drr_pred:.1f}</td>
                    <td style='text-align:right;padding:3px'>{drr_ref:.1f}</td>
                    <td style='text-align:right;padding:3px'>{drr_diff:.1f}</td></tr>
                <tr><td style='padding:3px'>C50 (dB)</td>
                    <td style='text-align:right;padding:3px'>{c50_pred:.1f}</td>
                    <td style='text-align:right;padding:3px'>{c50_ref:.1f}</td>
                    <td style='text-align:right;padding:3px'>{abs(c50_pred-c50_ref):.1f}</td></tr>
                <tr><td style='padding:3px'>C80 (dB)</td>
                    <td style='text-align:right;padding:3px'>{c80_pred:.1f}</td>
                    <td style='text-align:right;padding:3px'>{c80_ref:.1f}</td>
                    <td style='text-align:right;padding:3px'>{abs(c80_pred-c80_ref):.1f}</td></tr>
              </table>
            </div>"""
            display(HTML(html))

        with out_audio:
            # Audio players
            from IPython.display import display as _disp, HTML as _HTML
            _disp(_HTML("<b style='color:inherit'>Predicted RIR</b>"))
            _disp(ipd.Audio(rir_pred, rate=SR, autoplay=False))
            _disp(_HTML("<b style='color:inherit'>Reference RIR (Sabine model)</b>"))
            _disp(ipd.Audio(rir_ref, rate=SR, autoplay=False))

            # Side-by-side comparison plots
            import matplotlib
            try:
                get_ipython().run_line_magic('matplotlib', 'inline')
            except Exception:
                matplotlib.use("Agg")
            import matplotlib.pyplot as plt

            t_axis = np.arange(len(rir_pred)) / SR
            fig, axes = plt.subplots(2, 2, figsize=(12, 6))

            # Top-left: Predicted RIR waveform
            axes[0, 0].plot(t_axis, rir_pred, linewidth=0.4, color="#2196F3")
            axes[0, 0].set_title("Generated RIR — Predicted")
            axes[0, 0].set_xlabel("Time (s)")
            axes[0, 0].set_ylabel("Amplitude")

            # Top-right: Reference RIR waveform
            axes[0, 1].plot(t_axis, rir_ref, linewidth=0.4, color="#F44336")
            axes[0, 1].set_title("Reference RIR — Sabine Model")
            axes[0, 1].set_xlabel("Time (s)")
            axes[0, 1].set_ylabel("Amplitude")

            # Bottom-left: EDC comparison
            axes[1, 0].plot(t_axis, edc_pred_db, color="#2196F3",
                            linewidth=1.2, label="Predicted")
            axes[1, 0].plot(t_axis, edc_ref_db, color="#F44336",
                            linewidth=1.2, linestyle="--", label="Reference")
            axes[1, 0].axhline(-60, ls=":", color="#888", lw=0.8, label="-60 dB")
            axes[1, 0].set_title("Energy Decay Curve Comparison")
            axes[1, 0].set_xlabel("Time (s)")
            axes[1, 0].set_ylabel("Level (dB)")
            axes[1, 0].set_ylim(-80, 5)
            axes[1, 0].legend(fontsize=8)

            # Bottom-right: Metric bar chart
            metrics_names = ["RT60\n(s)", "DRR\n(dB)", "C50\n(dB)", "C80\n(dB)"]
            pred_vals = [rt60_pred, drr_pred, c50_pred, c80_pred]
            ref_vals = [rt60_ref, drr_ref, c50_ref, c80_ref]
            x_pos = np.arange(len(metrics_names))
            w = 0.35
            axes[1, 1].bar(x_pos - w/2, pred_vals, w, label="Predicted",
                           color="#2196F3", alpha=0.8)
            axes[1, 1].bar(x_pos + w/2, ref_vals, w, label="Reference",
                           color="#F44336", alpha=0.8)
            axes[1, 1].set_xticks(x_pos)
            axes[1, 1].set_xticklabels(metrics_names)
            axes[1, 1].set_title("Metrics Comparison")
            axes[1, 1].legend(fontsize=8)
            axes[1, 1].axhline(0, color="#888", lw=0.5)

            plt.tight_layout()
            plt.show()

    # Register click callback
    gen_btn = widgets.Button(description="▶ Generate RIR", button_style="success",
                             layout=Layout(width="180px", height="36px"))
    gen_btn.on_click(on_generate)

    title = widgets.HTML(
        "<h3 style='margin-bottom:4px;color:inherit'>"
        "🎤 Interactive RIR Demo</h3>"
        "<p style='color:inherit;opacity:0.7;font-size:12px'>"
        "Adjust sliders and click Generate.</p>"
    )

    ui = VBox([
        title,
        widgets.HTML("<b style='color:inherit'>Room Geometry</b>"),
        sl_L, sl_W, sl_H, sl_abs,
        widgets.HTML("<b style='color:inherit'>Source Position</b>"),
        sl_sx, sl_sy, sl_sh,
        gen_btn,
        widgets.HTML("<b style='color:inherit'>Output</b>"),
        out_metrics,
        out_audio,
    ])
    ipd.display(ui)
    # Trigger initial render
    on_generate(None)
    print("\n✓ Demo ready — adjust sliders or click Generate to synthesise a new RIR.")


# ── Auto-run ───────────────────────────────────────────────────────────
run_live_demo()
